# Figure 5: interaction classes and erythroid-associated modules (Norman et al.)

Run this notebook from the `figures/` folder after populating `data/` and `results/`
(see the repository README). Each section below reruns the relevant part of an analysis
notebook from the saved models; panels are written to `figures/panels/`
(`paths.FIGURES_DIR`).

Panel b shows the redundant, suppression and synergy arcs of the arc plot drawn below,
which shows all interaction classes (Fig. S13a).

In [ ]:
import os
import sys

# Repository root, so that paths.py (DATA_DIR / RESULTS_DIR / FIGURES_DIR) can be imported.
sys.path.insert(0, os.path.abspath('..'))
import paths

for d in (paths.DATA_DIR, paths.RESULTS_DIR, paths.FIGURES_DIR):
    print(d, '(found)' if os.path.isdir(d) else '(missing)')

## Norman CRISPRa screen

From [`scripts/norman/1_norman_analysis.ipynb`](../scripts/norman/1_norman_analysis.ipynb).

Loads the five SHERLOCK runs, the baseline runs and the GEARS runs from `results/norman_figure/`; any run that is missing is trained.

# Norman 2019 — Main Figure

Runs SHERLOCK (`l0_lambda=25, rank=6, synergy=True`) `N_RUNS` times on the held-out Norman split,
selects the best model by held-out combo ATE, and assembles a four-panel journal figure:

| Panel | Content |
|---|---|
| **(a)** | Counterfactual ATE across N runs — reproducibility & OOD generalisation |
| **(b)** | Single-perturbation latent UMAP (best model) |
| **(c)** | Parent-coefficient scatter (c₁ vs c₂) coloured by Norman GT class |
| **(d)** | GI classification AUC-ROC per class — SHERLOCK vs GEARS |

The notebook saves every intermediate result to `results/norman_figure/` and the final
composite figure to `results/norman_figure/main_figure.pdf`.

In [ ]:
import os, sys, math, random, textwrap
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr, bootstrap
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import torch

import sherlock as slk
from sherlock.tools._models import classify_gi

# ── Journal figure style ────────────────────────────────────────────────
RC = {
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':             8,
    'axes.labelsize':        8,
    'axes.titlesize':        9,
    'xtick.labelsize':       7,
    'ytick.labelsize':       7,
    'legend.fontsize':       6,
    'legend.title_fontsize': 7,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.linewidth':        0.75,
    'xtick.major.width':     0.75,
    'ytick.major.width':     0.75,
    'xtick.major.size':      3,
    'ytick.major.size':      3,
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
    'svg.fonttype': 'none',
}
mpl.rcParams.update(RC)

PPI = 96  # Figma/CSS px-per-inch — matches how pt-labeled SVGs get imported

# ── Colour palette ─────────────────────────────────────────────────────
CLR_SHERLOCK    = '#2166ac'
CLR_GEARS       = '#d73027'

# Keys match the exact strings in genetic_interactions.csv
PALETTE_GI = {
    'approximately additive':         '#4DAF4A',  # green
    'epistasis':                      '#FF7F00',  # orange
    'neomorphic':                     '#984EA3',  # purple
    'potentiation':                   '#F781BF',  # pink
    'redundant':                      '#A65628',  # brown
    'suppression':                    '#E41A1C',  # red
    'synergy (dissimilar phenotype)': '#377EB8',  # blue
    'synergy (similar phenotype)':    '#6BAED6',  # light blue
}

def _add_umap_axis_indicator(ax, x_label='UMAP1', y_label='UMAP2', length=0.15,
                              lw=1.0, color='0.15', fontsize=None):
    """Small L-shaped corner axis indicator (scanpy/Seurat style) — two short
    arrows from the bottom-left corner instead of full tick-labelled spines,
    used on UMAP panels where absolute coordinates are meaningless."""
    if fontsize is None:
        fontsize = mpl.rcParams['axes.labelsize']
    ax.annotate('', xy=(length, 0), xytext=(0, 0),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                 shrinkA=0, shrinkB=0))
    ax.annotate('', xy=(0, length), xytext=(0, 0),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                 shrinkA=0, shrinkB=0))
    ax.text(length / 2, -0.03, x_label, transform=ax.transAxes, fontsize=fontsize,
            ha='center', va='top', color=color)
    ax.text(-0.03, length / 2, y_label, transform=ax.transAxes, fontsize=fontsize,
            ha='right', va='center', color=color, rotation=90)


# ── Constants ──────────────────────────────────────────────────────────
N_RUNS    = 5
SEED_BASE = 0
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR   = Path(f'{paths.RESULTS_DIR}/norman_figure')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}  |  N_RUNS: {N_RUNS}  |  Output: {OUT_DIR}')

## 1. Load and preprocess Norman 2019

In [ ]:
data = sc.read_h5ad(f'{paths.DATA_DIR}/norman/Norman_2019.h5ad')
data.X = data.layers['counts'].copy()

slk.pp.slk_prepare_data(
    data,
    pert_key='perturbation_name',
    ntc_label='control',
    treatment_key=None,
    inplace=True,
)

p_key = slk.configs.get_config('pert_key')
NTC   = slk.configs.get_config('ntc_label')

all_perts    = data.obs[p_key].unique()
single_perts = [p for p in all_perts if '+' not in str(p) and p != NTC]
combo_perts  = [p for p in all_perts if '+' in str(p)]
print(f'Cells: {data.n_obs}  |  Genes: {data.n_vars}')
print(f'Single perts: {len(single_perts)}  |  Combo perts: {len(combo_perts)}')

In [ ]:
# Gene filtering: keep genes with mean count > 0.5 UMI or that are pert targets
pert_genes = set()
for p in all_perts:
    if p != NTC:
        for g in str(p).split('+'):
            pert_genes.add(g.strip())

mean_expr = np.array(data.X.mean(axis=0)).flatten()
expr_mask = mean_expr > 0.5
pert_mask = data.var_names.isin(pert_genes)
keep_mask = expr_mask | pert_mask
data = data[:, keep_mask].copy()

print(f'Kept {data.n_vars} genes  '
      f'({expr_mask[keep_mask].sum()} expr + '
      f'{(pert_mask[keep_mask] & ~expr_mask[keep_mask]).sum()} pert-only)')

In [ ]:
# Observed treatment effects (log2-CPM vs NTC baseline)
slk.pp.treat_effect(
    data,
    label_col='pert',
    control_label='NTC',
    method='perturbseq',
    inplace=True,
    key='treat_effect',
)
te = data.uns['treat_effect']
print(f'treat_effect: {te.n_obs} perturbations × {te.n_vars} genes')

## 2. Train/val split — 20% of each Norman GT class held out

In [ ]:
def canon_pair(s):
    s = str(s)
    if '+' not in s:
        return s
    a, b = s.split('+', 1)
    return '+'.join(sorted([a.strip(), b.strip()]))

random.seed(42)
gi_labels = pd.read_csv(f'{paths.DATA_DIR}/norman/genetic_interactions.csv')
gi_labels['combination'] = gi_labels['combination'].apply(canon_pair)
gi_labels['genetic_interaction'] = gi_labels['genetic_interaction'].str.strip()

val_combos = set()
for cls, grp in gi_labels.groupby('genetic_interaction'):
    n_val = math.ceil(0.2 * len(grp))
    val_combos.update(random.sample(list(grp['combination']), n_val))

val_mask   = data.obs[p_key].apply(lambda p: ('+' in str(p)) and (canon_pair(p) in val_combos))
data_train = data[~val_mask].copy()

print(f'Total cells: {data.n_obs}  |  Training: {data_train.n_obs}')
print(f'GT pairs: {len(gi_labels)}  |  Held-out val combos: {len(val_combos)}')

## 3. Multi-run training

Trains SHERLOCK `N_RUNS` times with identical settings. After each run, computes ATE on
the full data (including held-out combos). The best run by held-out combo ATE is used for
UMAP, synergy analysis, and GI classification.

In [ ]:
def _ate_r(pred_df, obs_df, idx):
    idx = [p for p in idx if p in pred_df.index and p in obs_df.index]
    if len(idx) < 2:
        return float('nan')
    x = pred_df.loc[idx].values.ravel()
    y = obs_df.loc[idx].values.ravel()
    v = np.isfinite(x) & np.isfinite(y)
    return float(pearsonr(x[v], y[v])[0]) if v.sum() > 2 else float('nan')


def _effect_size_r(pred_df, obs_df):
    common = pred_df.index.intersection(obs_df.index)
    if len(common) < 3:
        return float('nan')
    p_norms = pred_df.loc[common].apply(np.linalg.norm, axis=1).values
    o_norms = obs_df.loc[common].apply(np.linalg.norm, axis=1).values
    v = np.isfinite(p_norms) & np.isfinite(o_norms)
    return float(pearsonr(p_norms[v], o_norms[v])[0]) if v.sum() > 2 else float('nan')


def _w_coact_matrix(W, threshold=0.5):
    """(P, P) cosine-normalized perturbation co-activation matrix. Invariant
    to permutation of the K gate dimensions, so it can be compared directly
    across independently trained runs (unlike the raw W)."""
    W_bin = (W > threshold).astype(float)
    dot   = W_bin @ W_bin.T
    norms = np.sqrt(np.diag(dot))
    denom = np.outer(norms, norms) + 1e-8
    return dot / denom


def _flat_triu(M):
    idx = np.triu_indices(M.shape[0], k=1)
    return M[idx]


def _loo_spearman_df(mats_by_run, metric_name):
    """Spearman r between each run's (P, P) matrix and the elementwise mean of
    every OTHER run's matrix (leave-one-out) — one row per run (n total).
    Unlike the C(n, 2) all-pairs comparisons, these n points each depend on a
    distinct held-out run, so they behave like an i.i.d. sample for
    bootstrapping instead of n(n-1)/2 non-independent pairs drawn from only
    n underlying runs."""
    rows = []
    keys = sorted(mats_by_run)
    for k in keys:
        others = [mats_by_run[j] for j in keys if j != k]
        if not others:
            continue
        mean_other = np.mean(others, axis=0)
        vi, vj = _flat_triu(mats_by_run[k]), _flat_triu(mean_other)
        valid  = np.isfinite(vi) & np.isfinite(vj)
        r = float(spearmanr(vi[valid], vj[valid])[0]) if valid.sum() > 2 else float('nan')
        rows.append({'metric': metric_name, 'run': k, 'r': r})
    return pd.DataFrame(rows)


def _compute_ate_metrics(model, data, val_combos):
    """Compute all ATE breakdown metrics for a model on full data."""
    ate  = model.predict_counterfactual_effects(
        data, n_centroids=25, observed_effect=data.uns['treat_effect'], approximate=False
    )
    pred = ate['predicted_df']
    obs  = ate['observed_df']
    singles = [p for p in pred.index if '+' not in str(p)]
    combos  = [p for p in pred.index if '+' in str(p) and canon_pair(p) not in val_combos]
    held    = [p for p in pred.index if canon_pair(p) in val_combos]
    return ate, pred, obs, {
        'ate_all':    ate['corr'],
        'ate_single': _ate_r(pred, obs, singles),
        'ate_combo':  _ate_r(pred, obs, combos),
        'ate_held':   _ate_r(pred, obs, held),
    }


def _cache_fresh(cache_path, *source_paths):
    """True if cache_path exists and is at least as new as every path in
    source_paths (files, or directories checked recursively). Retraining a
    model should invalidate every cache derived from it; comparing mtimes
    lets that happen automatically instead of a cache silently outliving the
    checkpoint(s) it was computed from."""
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    cache_mtime = cache_path.stat().st_mtime
    for src in source_paths:
        src = Path(src)
        if not src.exists():
            return False
        if src.is_dir():
            sub_paths = list(src.rglob('*'))
            if not sub_paths:
                continue
            src_mtime = max(p.stat().st_mtime for p in sub_paths)
        else:
            src_mtime = src.stat().st_mtime
        if src_mtime > cache_mtime:
            return False
    return True


# ── Paths ──────────────────────────────────────────────────────────────
MODELS_DIR        = OUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUN_METRICS_PATH  = OUT_DIR / 'run_metrics.csv'
RUN_MATRICES_PATH = OUT_DIR / 'run_matrices.npz'
PAIRWISE_PATH     = OUT_DIR / 'pairwise_repro.csv'
checkpoints       = {i + 1: MODELS_DIR / f'run_{i + 1}.pth' for i in range(N_RUNS)}

if RUN_METRICS_PATH.exists():
    _saved = pd.read_csv(RUN_METRICS_PATH)
    existing_metrics = {int(r['run']): r for r in _saved.to_dict('records')}
else:
    existing_metrics = {}

if RUN_MATRICES_PATH.exists():
    _saved_mats  = np.load(RUN_MATRICES_PATH)
    rho_by_run   = {int(k.split('_')[-1]): _saved_mats[k] for k in _saved_mats.files if k.startswith('rho_corr_')}
    wcoact_by_run = {int(k.split('_')[-1]): _saved_mats[k] for k in _saved_mats.files if k.startswith('wcoact_')}
else:
    rho_by_run, wcoact_by_run = {}, {}

# ── Training / loading loop ────────────────────────────────────────────
for run_num in range(1, N_RUNS + 1):
    ckpt_path    = checkpoints[run_num]
    already_done = (run_num in existing_metrics) and ckpt_path.exists()

    if already_done:
        m = existing_metrics[run_num]
        print(f'Run {run_num}: loaded  '
              f"ate_all={m['ate_all']:.4f}  ate_single={m.get('ate_single', float('nan')):.4f}  "
              f"ate_combo={m.get('ate_combo', float('nan')):.4f}  ate_held={m['ate_held']:.4f}")
        if run_num not in rho_by_run or run_num not in wcoact_by_run:
            _model = slk.tl.load_results(ckpt_path)
            slk.tl.eval_single(_model, data_train, obsm_key='_z_tmp', uns_key='_tmp', device=DEVICE)
            _tmp = data_train.uns['_tmp']
            if _tmp.get('rho_corr') is not None:
                rho_by_run[run_num] = np.asarray(_tmp['rho_corr'], dtype=float)
            if _tmp.get('W') is not None:
                wcoact_by_run[run_num] = _w_coact_matrix(np.asarray(_tmp['W'], dtype=float))
        continue

    print(f'\n── Training run {run_num}/{N_RUNS} ──────────────────────────')
    out = slk.tl.run_single(
        data_train,
        model='vae',
        l0_lambda=1e-1, #20
        rank=16,
        combinatorial=True,
        synergy=True,
        synergy_rank=6,
        shuffle=True,
        num_workers=0,
        device=DEVICE,
        num_epochs=500,
        validate_every=50,
        patience=30, #20?
        n_epochs_l0_warmup=300,
        seed=SEED_BASE + run_num - 1,
        debug=False,
        use_de_align_loss=False,
        de_align_lambda=10,    
        de_align_direction_lambda=0,
        de_align_n_bg=512, #256 prob works
        de_align_exclude=[], 
        use_go_prior=False,
    )
    slk.tl.save_results(out, ckpt_path)

    model = out['model']
    _, pred, obs, ate_metrics = _compute_ate_metrics(model, data, val_combos)

    slk.tl.eval_single(model, data_train, obsm_key='_z_tmp', uns_key='_tmp', device=DEVICE)
    tmp   = data_train.uns['_tmp']
    W_mat = tmp.get('W')
    if tmp.get('rho_corr') is not None:
        rho_by_run[run_num] = np.asarray(tmp['rho_corr'], dtype=float)
    if W_mat is not None:
        wcoact_by_run[run_num] = _w_coact_matrix(np.asarray(W_mat, dtype=float))

    m = {
        'run':           run_num,
        **ate_metrics,
        'effect_size_r': _effect_size_r(pred, obs),
    }
    existing_metrics[run_num] = m
    pd.DataFrame(list(existing_metrics.values())).sort_values('run').to_csv(
        RUN_METRICS_PATH, index=False,
    )
    print(f"  ate_all={m['ate_all']:.4f}  ate_single={m['ate_single']:.4f}  "
          f"ate_combo={m['ate_combo']:.4f}  ate_held={m['ate_held']:.4f}")

np.savez(
    RUN_MATRICES_PATH,
    **{f'rho_corr_{k}': v for k, v in rho_by_run.items()},
    **{f'wcoact_{k}':   v for k, v in wcoact_by_run.items()},
)

# ── Cross-run reproducibility (rho_corr, W coactivation) ────────────────
# For each run, Spearman r between flattened upper-triangles of its rho_corr
# (learned perturbation covariance) / W-coactivation matrix and the mean of
# the OTHER runs' matrices (leave-one-out). This gives n independent points
# (one per run) instead of the C(n, 2) all-pairs comparisons, which reuse
# each run across multiple pairs and so are not i.i.d. even though there are
# more of them. effect_size_r stays a per-run metric since it measures
# within-run predictive accuracy, not cross-run reproducibility.
runs_df = pd.read_csv(RUN_METRICS_PATH)

pairwise_df = pd.concat([
    _loo_spearman_df(rho_by_run, 'rho_corr_r'),
    _loo_spearman_df(wcoact_by_run, 'w_coact_r'),
], ignore_index=True)
pairwise_df.to_csv(PAIRWISE_PATH, index=False)

for _metric, _grp in pairwise_df.groupby('metric'):
    _vals = _grp['r'].dropna().values
    if len(_vals) >= 3:
        _boot = bootstrap((_vals,), np.mean, confidence_level=0.95,
                           method='percentile', random_state=0)
        print(f'{_metric}: mean r = {_vals.mean():.4f}  '
              f'95% CI [{_boot.confidence_interval.low:.4f}, {_boot.confidence_interval.high:.4f}]  '
              f'(n_runs={len(_vals)})')
    elif len(_vals) == 1:
        print(f'{_metric}: r = {_vals[0]:.4f}  '
              f'(only 1 run — need >=3 runs for a bootstrap CI)')
    else:
        print(f'{_metric}: no valid runs.')

# ── Select and load best model ─────────────────────────────────────────
runs_df  = runs_df[runs_df['run'].map(lambda r: checkpoints.get(int(r), Path()).exists())]
best_run = int(runs_df.loc[runs_df['ate_held'].idxmax(), 'run'])
best_ate = float(runs_df['ate_held'].max())

print(f'\nBest run: {best_run}  (ate_held = {best_ate:.4f})')
best_model = slk.tl.load_results(checkpoints[best_run])
runs_df

## 3b. Other benchmark methods (cVAE, sVAE, contrastiveVI, scgen)

Same hyperparameters/model kwargs as `benchmarking_combinatorial.ipynb`'s `METHODS` list, trained
on the same held-out combinatorial split (`data_train`/`val_combos`). `sherlock_synergy` isn't
listed separately since the SHERLOCK run above already trains with `synergy=True`. Each
method/run is checkpointed independently (as in `benchmarking.ipynb`), so a re-executed notebook
reloads finished runs instead of retraining.

**Combos are atomic labels, not compositional.** Unlike SHERLOCK (`combinatorial=True`, which
splits `"A+B"` into components and sums their learned shifts — the mechanism that lets it
extrapolate to held-out combos at all), these baselines are run with `combinatorial=False`:
`"A+B"` gets its own independent embedding, exactly like a third, unrelated perturbation — no
sharing with `"A"` or `"B"`. That means a combo the model never saw in training has *no* learned
representation whatsoever, so held-out ATE is undefined for these methods by construction (not
just poor). Evaluation therefore uses `data_train` (not the full `data`) as the ATE prediction
input: `predict_counterfactual_effects` rebuilds its own perturbation index internally from
whatever `adata` it's given, so evaluating on the full `data` here would silently reintroduce the
held-out combos into that index and misalign every perturbation sorted after them relative to the
model's trained embedding table — evaluating on `data_train` keeps the eval-time and train-time
indices identical and simply leaves held-out combos absent (→ `ate_held` = NaN via the existing
`_ate_r` NaN-on-empty-index handling), correctly and safely. In Panel A below, that means the
`ATE (held-out)` box is populated by SHERLOCK only.

In [ ]:
# ── Other benchmark methods (cVAE, sVAE, contrastiveVI, scgen) ────────────
OTHER_METHODS = [
    ('cvae',          'cVAE',          {'combinatorial': False}),
    ('svae',          'sVAE',          {'sparse_mask_penalty': 10.0, 'combinatorial': False}),
    ('contrastivevi', 'contrastiveVI', {'combinatorial': False}),
    ('scgen',         'scgen',         {}),
]
OTHER_EPOCHS         = 200
OTHER_BATCH_SIZE     = 4096
OTHER_LATENT_DIM     = 16
OTHER_PATIENCE       = 20
OTHER_VALIDATE_EVERY = 50

BENCH_MODELS_DIR   = OUT_DIR / 'benchmark_models'
BENCH_MODELS_DIR.mkdir(parents=True, exist_ok=True)
BENCH_METRICS_PATH = OUT_DIR / 'benchmark_run_metrics.csv'

if BENCH_METRICS_PATH.exists():
    bench_records = pd.read_csv(BENCH_METRICS_PATH).to_dict('records')
else:
    bench_records = []
bench_done = {(r['method'], int(r['run'])) for r in bench_records}

for model_key, display_name, extra_kwargs in OTHER_METHODS:
    method_dir = BENCH_MODELS_DIR / display_name
    method_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}\n{display_name}  ({N_RUNS} runs, {OTHER_EPOCHS} epochs)\n{'='*60}")

    for run_num in range(1, N_RUNS + 1):
        if (display_name, run_num) in bench_done:
            print(f'  run {run_num}/{N_RUNS}: metrics already cached, skipping.')
            continue

        ckpt_path = method_dir / f'run_{run_num}.pth'
        if ckpt_path.exists():
            print(f'  run {run_num}/{N_RUNS}: loading checkpoint {ckpt_path}')
            model = slk.tl.load_results(ckpt_path)
        else:
            print(f'  run {run_num}/{N_RUNS}: training...')
            out = slk.tl.run(
                data_train,
                model=model_key,
                batch_size=OTHER_BATCH_SIZE,
                num_epochs=OTHER_EPOCHS,
                device=DEVICE,
                latent_dim=OTHER_LATENT_DIM,
                patience=OTHER_PATIENCE,
                validate_every=OTHER_VALIDATE_EVERY,
                seed=SEED_BASE + run_num - 1,
                **extra_kwargs,
            )
            model = out['model']
            slk.tl.save_results(out, ckpt_path)

        # data_train (not data) — see markdown above: these models treat every
        # combo as an atomic label, so evaluating on data_train keeps the
        # eval-time perturbation index identical to the trained one and
        # leaves held-out combos correctly absent (ate_held -> NaN) instead
        # of silently misaligning indices for everything else.
        _, _, _, ate_metrics = _compute_ate_metrics(model, data_train, val_combos)
        record = {'method': display_name, 'run': run_num, **ate_metrics}
        bench_records.append(record)
        bench_done.add((display_name, run_num))
        pd.DataFrame(bench_records).to_csv(BENCH_METRICS_PATH, index=False)
        print(f"    ate_all={record['ate_all']:.4f}  ate_single={record['ate_single']:.4f}  "
              f"ate_combo={record['ate_combo']:.4f}  ate_held={record['ate_held']:.4f}")

bench_runs_df = pd.DataFrame(bench_records)
print(f'\nOther-method benchmark done: {len(bench_runs_df)} (method, run) rows.')
bench_runs_df

## 4. Best model — latent embedding & GI scoring

In [ ]:
SYN_PATH = OUT_DIR / 'syn_best.csv'

# Always run eval to populate data.obsm['z'] (fast; needed for UMAP + get_gi)
model = best_model
slk.tl.eval_single(model, data, obsm_key='z', uns_key='results', device=DEVICE)

if _cache_fresh(SYN_PATH, checkpoints[best_run]):
    print(f'Loading cached GI scores from {SYN_PATH}')
    syn = pd.read_csv(SYN_PATH, index_col=0)
else:
    syn = model.get_gi(data, min_cells=5)

    # Canonicalise index and annotate with Norman GT labels
    syn.index = (syn['pert_a'].astype(str) + '+' + syn['pert_b'].astype(str)).map(canon_pair)
    gi_series = gi_labels.set_index('combination')['genetic_interaction']
    syn['gt_gi'] = syn.index.map(gi_series)

    gt_combos = set(gi_labels['combination'])
    fit_mask  = syn.index.to_series().isin(gt_combos)
    syn = classify_gi(syn, fit_mask=fit_mask, inplace=True)

    syn.to_csv(SYN_PATH)
    print(f'GI scores saved to {SYN_PATH}')

# Ensure gi_series is always defined for downstream cells
gi_series = gi_labels.set_index('combination')['genetic_interaction']

print(f'GI rows: {len(syn)}  |  GT-matched: {syn["gt_gi"].notna().sum()}  |  '
      f'Classified: {syn["predicted_gi"].notna().sum()}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def classify_and_report_gi(
    syn_df,
    gt_path=f"{paths.DATA_DIR}/norman/genetic_interactions.csv",
    pred_col="predicted_gi",
    inplace=True,
    figsize=(8, 6),
):
    def canon_pair(s):
        s = str(s)
        if "+" not in s:
            return s
        a, b = s.split("+", 1)
        return "+".join(sorted([a.strip(), b.strip()]))

    if not os.path.exists(gt_path):
        raise FileNotFoundError(f"Expected {gt_path} for Norman labels.")

    gt = pd.read_csv(gt_path)
    gt["combination"] = gt["combination"].map(canon_pair)
    gt["genetic_interaction"] = gt["genetic_interaction"].astype(str).str.strip()

    eval_combinations = set(gt["combination"])
    syn_combinations = (
        syn_df["pert_a"].astype(str) + "+" + syn_df["pert_b"].astype(str)
    ).map(canon_pair)

    classified = slk.tl.classify_gi(
        syn_df,
        pred_col=pred_col,
        # fit_mask=syn_combinations.isin(eval_combinations),
        inplace=inplace,
    )

    score_df = classified.copy()
    score_df["combination"] = syn_combinations
    score_df = score_df.merge(
        gt[["combination", "genetic_interaction"]],
        on="combination",
        how="inner",
    )
    score_df = score_df.loc[score_df[pred_col].notna()].copy()

    y_true = score_df["genetic_interaction"].astype(str)
    y_pred = score_df[pred_col].astype(str)
    labels = sorted(y_true.unique())

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", labels=labels)

    print(f"Evaluated {len(score_df)} labeled interactions.")
    print(f"Accuracy:    {acc:.3f}")
    print(f"Macro F1:    {macro_f1:.3f}")
    print(f"Weighted F1: {weighted_f1:.3f}")
    print()
    print("Classification report:")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Norman label")
    ax.set_title("GI score confusion matrix")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return classified

classify_and_report_gi(syn)
# gears_gi = classify_and_report_gi(gears_gi)


In [ ]:
UMAP_PATH     = OUT_DIR / 'umap_singlepert.npz'
UMAP_ALL_PATH = OUT_DIR / 'umap_allpert.npz'

single_mask = (
    (data.obs[p_key] != NTC) &
    (~data.obs['perturbation_name'].astype(str).str.contains(r'\+', regex=True))
)
sub = data[single_mask].copy()

# All non-NTC cells (singles + combos) for panel C
all_mask = data.obs[p_key] != NTC
all_sub  = data[all_mask].copy()
all_sub.obs['pert_type'] = np.where(
    all_sub.obs[p_key].astype(str).str.contains(r'\+', regex=True),
    'combination', 'single',
)

if _cache_fresh(UMAP_PATH, checkpoints[best_run]):
    print(f'Loading cached single-pert UMAP from {UMAP_PATH}')
    _u = np.load(UMAP_PATH, allow_pickle=True)
    sub.obsm['X_umap'] = _u['coords']
else:
    sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
    sc.tl.umap(sub)
    np.savez(UMAP_PATH, coords=sub.obsm['X_umap'])
    print(f'Single-pert UMAP saved to {UMAP_PATH}')

if _cache_fresh(UMAP_ALL_PATH, checkpoints[best_run]):
    print(f'Loading cached all-pert UMAP from {UMAP_ALL_PATH}')
    _u2 = np.load(UMAP_ALL_PATH, allow_pickle=True)
    all_sub.obsm['X_umap'] = _u2['coords']
else:
    sc.pp.neighbors(all_sub, n_neighbors=15, use_rep='z')
    sc.tl.umap(all_sub)
    np.savez(UMAP_ALL_PATH, coords=all_sub.obsm['X_umap'])
    print(f'All-pert UMAP saved to {UMAP_ALL_PATH}')

print(f'Single-pert cells: {sub.n_obs}  |  All-pert cells: {all_sub.n_obs}')

## 5. GEARS multi-run training & GI classification

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, silhouette_score, silhouette_samples,
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score,
)

GI_METRICS = [
    'additive_residual_norm', 'dominance',
    'parent_dcor_diff', 'equality_contribution', 'singles_similarity',
    'norm_ab_over_additive', 'norm_fit_over_ab', 'magnitude',
    'model_fit_dcor', 'norm_fit', 'signed_residual_alignment',
]


# ── Merge the two synergy sub-types into a single 'synergy' class ───────
# Norman splits synergy into (similar phenotype) / (dissimilar phenotype). For
# the GI classification figure we score them as ONE class. This must happen
# BEFORE metrics are computed: the per-class F1/AUC of two one-vs-rest classes
# cannot be averaged into a merged class after the fact.
# Only the *metrics* are merged — the returned `work` frame keeps the original
# 8 labels, so Panel D's UMAP colouring is unchanged.
MERGE_SYNERGY = True
_SYNERGY_ALIASES = {
    'synergy (similar phenotype)':    'synergy',
    'synergy (dissimilar phenotype)': 'synergy',
}

def _merge_syn(s):
    """Collapse syn-sim / syn-dis -> 'synergy' (no-op if MERGE_SYNERGY is False)."""
    s = s.astype(str)
    return s.replace(_SYNERGY_ALIASES) if MERGE_SYNERGY else s


def evaluate_gi_classification(df, model_name, gi_series):
    """Classify GI for all combinations; evaluate against GT where available."""
    work = df.copy()
    if 'pert_a' in work.columns and 'pert_b' in work.columns:
        work.index = (work['pert_a'].astype(str) + '+' + work['pert_b'].astype(str)).map(canon_pair)
    else:
        work.index = work.index.map(canon_pair)

    work['gt_gi'] = work.index.map(gi_series)

    # Classify ALL rows with finite metrics (no fit_mask) so unlabelled combos get predictions too
    work = classify_gi(work, inplace=True)

    eval_df = work[work['gt_gi'].notna() & work['predicted_gi'].notna()].copy()
    y_true  = _merge_syn(eval_df['gt_gi'])
    y_pred  = _merge_syn(eval_df['predicted_gi'])
    labels  = sorted(set(y_true))

    per_class = []
    for cls in labels:
        tb = (y_true == cls).astype(int)
        pb = (y_pred == cls).astype(int)
        n_pos = int(tb.sum())
        auc = roc_auc_score(tb, pb) if 0 < n_pos < len(tb) else np.nan
        per_class.append({
            'model': model_name, 'class': cls, 'n_positive': n_pos,
            'f1': f1_score(tb, pb, zero_division=0), 'auc_roc': auc,
        })

    summary = {
        'model':       model_name,
        'n_eval':      len(eval_df),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'ari':         adjusted_rand_score(y_true, y_pred),
        'ami':         adjusted_mutual_info_score(y_true, y_pred),
        'nmi':         normalized_mutual_info_score(y_true, y_pred),
    }
    return work, pd.DataFrame(per_class), summary


slk_classified, slk_class_df, slk_summary = evaluate_gi_classification(syn, 'SHERLOCK', gi_series)

In [ ]:
# ── GEARS multi-run training ─────────────────────────────────────────────
# Mirrors norman_run.ipynb: trains GEARS on data_train (held-out val_combos
# excluded, same split SHERLOCK uses) and predicts on gears_combos = all
# combinations observed in the full `data` (train-visible + held-out), i.e.
# train_gears(data_train, combos=get_observed_combinations(data, ...)).
# Runs N_RUNS times for CI in panel E; each run's model checkpoint, raw
# predictions, and classified metrics are cached independently so a run that
# already finished (even if a later run was interrupted) is never redone.
import sherlock.tools._gears as gears_tools
slk.tl.run_gears_gi              = gears_tools.run_gears_gi
slk.tl.get_observed_combinations = gears_tools.get_observed_combinations

GEARS_MODELS_DIR = OUT_DIR / 'gears_models'
GEARS_RUNS_DIR   = OUT_DIR / 'gears_runs'
GEARS_MODELS_DIR.mkdir(parents=True, exist_ok=True)
GEARS_RUNS_DIR.mkdir(parents=True, exist_ok=True)
GEARS_CLS_CACHE  = OUT_DIR / 'gears_per_run_class.csv'
GEARS_SUMM_CACHE = OUT_DIR / 'gears_per_run_summ.csv'

gears_combos = slk.tl.get_observed_combinations(data, pert_key=p_key, ntc_label=NTC)
print(f'GEARS combos (observed in full data, train + held-out): {len(gears_combos)}')
gears_singles = gears_tools.get_single_perturbations(data, pert_key=p_key, ntc_label=NTC)
print(f'GEARS singles (observed in full data): {len(gears_singles)}')

from scipy.sparse import csr_matrix, issparse

data_train_gears = data_train.copy()
_X_gears = data_train_gears.X
if issparse(_X_gears):
    _X_gears = _X_gears.toarray()
_X_gears = 1e4 * _X_gears / np.sum(_X_gears, axis=1, keepdims=True)
_X_gears = np.log2(_X_gears + 1)
data_train_gears.X = csr_matrix(_X_gears)  # GEARS assumes sparse .X, calls .toarray() internally

if GEARS_CLS_CACHE.exists() and GEARS_SUMM_CACHE.exists():
    _gears_run_class_df = pd.read_csv(GEARS_CLS_CACHE)
    _gears_run_summ_df  = pd.read_csv(GEARS_SUMM_CACHE)
else:
    _gears_run_class_df = pd.DataFrame(columns=['run'])
    _gears_run_summ_df  = pd.DataFrame(columns=['run'])
_gears_done_runs = set(_gears_run_summ_df['run'].unique().tolist())

gears_df = gears_classified = gears_class_df = gears_summary = None

for _rn in range(1, N_RUNS + 1):
    _pred_path = GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv'
    _ckpt_dir  = GEARS_MODELS_DIR / f'run_{_rn}'
    _has_ckpt  = (_ckpt_dir / 'config.pkl').exists()
    _preds_fresh = _has_ckpt and _cache_fresh(_pred_path, _ckpt_dir)

    if _rn in _gears_done_runs and _preds_fresh:
        print(f'GEARS run {_rn}/{N_RUNS}: metrics already cached and up to date, skipping.')
        gears_df = pd.read_csv(_pred_path)
        continue

    print(f'\n── GEARS run {_rn}/{N_RUNS} ──────────────────────────')
    _seed = SEED_BASE + _rn - 1
    random.seed(_seed)
    np.random.seed(_seed)
    torch.manual_seed(_seed)

    if _preds_fresh:
        print(f'  loading cached predictions from {_pred_path}')
        _gears_df_r = pd.read_csv(_pred_path)
        _ate_metrics_r = None
    else:
        _gears_run = slk.tl.run_gears_gi(
            data_train_gears,
            combos=gears_combos,
            data_dir=f'{paths.RESULTS_DIR}/gears_data',
            dataset_name='norman_figure_gears',
            pert_key=p_key,
            ntc_label=NTC,
            split='all',
            seed=_seed,
            batch_size=256,
            test_batch_size=1024,
            epochs=20,
            load_path=str(_ckpt_dir) if _has_ckpt else None,
            model_path=None if _has_ckpt else str(_ckpt_dir),
            verbose=True,
            quiet_steps=True,
            device=str(DEVICE),
        )
        _gears_df_r = _gears_run.gi if _gears_run.gi is not None else _gears_run.predictions
        _gears_df_r.to_csv(_pred_path, index=False)
        _ate_metrics_r = _gears_run.compute_ate(
            gears_singles + gears_combos, data.uns['treat_effect'], val_combos,
            already_normalized=True, verbose=False,
        )

    _classified_r, _cls_r, _summ_r = evaluate_gi_classification(_gears_df_r, 'GEARS', gi_series)
    _summ_r = {
        **_summ_r,
        **(_ate_metrics_r or {'ate_all': float('nan'), 'ate_single': float('nan'),
                               'ate_combo': float('nan'), 'ate_held': float('nan')}),
    }
    _cls_r = _cls_r.copy()
    _cls_r['run'] = _rn

    _gears_run_class_df = pd.concat([_gears_run_class_df, _cls_r], ignore_index=True)
    _gears_run_summ_df  = pd.concat(
        [_gears_run_summ_df, pd.DataFrame([{**_summ_r, 'run': _rn}])], ignore_index=True,
    )
    _gears_run_class_df.to_csv(GEARS_CLS_CACHE, index=False)
    _gears_run_summ_df.to_csv(GEARS_SUMM_CACHE, index=False)
    _gears_done_runs.add(_rn)

    gears_df, gears_classified, gears_class_df, gears_summary = (
        _gears_df_r, _classified_r, _cls_r, _summ_r,
    )
    print(f"  run {_rn}: f1_weighted={_summ_r['f1_weighted']:.4f}  "
          f"ate_all={_summ_r['ate_all']:.4f}  "
          f"ari={_summ_r['ari']:.4f}  ami={_summ_r['ami']:.4f}")

# If every run was already cached, reload the last run's predictions so
# downstream single-instance uses (UMAP fit, summary table) have a
# representative GEARS model.
if gears_classified is None:
    gears_df = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{N_RUNS}.csv')
    gears_classified, gears_class_df, gears_summary = evaluate_gi_classification(
        gears_df, 'GEARS', gi_series,
    )

print(f'\nGEARS per-run metrics:')
print(_gears_run_summ_df[['run', 'ate_all', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

class_df   = pd.concat([slk_class_df, gears_class_df], ignore_index=True)
summary_df = pd.DataFrame([slk_summary, gears_summary])
class_df.to_csv(OUT_DIR / 'gi_classification.csv', index=False)

for name, clf in [('SHERLOCK', slk_classified), ('GEARS', gears_classified)]:
    n_pred = clf['predicted_gi'].notna().sum()
    n_nan  = clf['predicted_gi'].isna().sum()
    print(f'{name}: {n_pred} predicted, {n_nan} NaN (should be 0 if all metrics finite)')

print(summary_df.to_string(index=False))

In [ ]:
# ── Per-run GI classification for SHERLOCK — used for CI in panel E ─────
# Runs get_gi + classify_gi on every checkpoint and caches results.
# Cache filename encodes the labelling, so flipping MERGE_SYNERGY busts the
# cache automatically instead of silently reusing rows with the old classes.
_suffix     = '_synmerged' if MERGE_SYNERGY else ''
_cls_cache  = OUT_DIR / f'gi_per_run_class{_suffix}.csv'
_summ_cache = OUT_DIR / f'gi_per_run_summ{_suffix}.csv'

if _cache_fresh(_cls_cache, *checkpoints.values()) and _cache_fresh(_summ_cache, *checkpoints.values()):
    print('Loading cached per-run GI metrics...')
    _run_class_df = pd.read_csv(_cls_cache)
    _run_summ_df  = pd.read_csv(_summ_cache)
else:
    _cls_rows, _summ_rows = [], []
    for _rn in range(1, N_RUNS + 1):
        _ckpt = checkpoints[_rn]
        if not _ckpt.exists():
            print(f'  Run {_rn}: checkpoint not found, skipping.')
            continue
        print(f'  Run {_rn}/{N_RUNS}...', end=' ', flush=True)
        _m = slk.tl.load_results(_ckpt)
        slk.tl.eval_single(_m, data, obsm_key='z', uns_key='results', device=DEVICE)
        _syn_r = _m.get_gi(data, min_cells=5)
        _syn_r.index = (
            _syn_r['pert_a'].astype(str) + '+' + _syn_r['pert_b'].astype(str)
        ).map(canon_pair)
        _syn_r['gt_gi'] = _syn_r.index.map(gi_series)
        _fit_r = _syn_r.index.to_series().isin(set(gi_labels['combination']))
        _syn_r  = classify_gi(_syn_r, fit_mask=_fit_r, inplace=True)
        _, _cls_r, _summ_r = evaluate_gi_classification(_syn_r, 'SHERLOCK', gi_series)
        _cls_r['run']  = _rn
        _summ_r['run'] = _rn
        _cls_rows.append(_cls_r)
        _summ_rows.append(_summ_r)
        print('done')
    _run_class_df = pd.concat(_cls_rows, ignore_index=True)
    _run_summ_df  = pd.DataFrame(_summ_rows)
    _run_class_df.to_csv(_cls_cache, index=False)
    _run_summ_df.to_csv(_summ_cache, index=False)
    print(f'Per-run GI metrics cached.')

# ── Rebuild GEARS per-run metrics under the merged-synergy labelling ─────
# GEARS' cached per-class/summary CSVs were computed with the 8-class labelling.
# Recompute them from the cached RAW predictions (gears_runs/gears_gi_run*.csv)
# so both models are scored with the same merged classes. No GEARS retraining and
# no cache files are overwritten. The ate_* columns are carried over from the
# on-disk summary cache: they are unaffected by the GI relabelling, and
# compute_ate() would need the GEARS model object, which we do not reload.
_g_cls_rows, _g_summ_rows = [], []
for _rn in range(1, N_RUNS + 1):
    _g_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv')
    _, _gc_r, _gs_r = evaluate_gi_classification(_g_raw, 'GEARS', gi_series)
    _gc_r = _gc_r.copy()
    _gc_r['run'] = _rn
    _g_cls_rows.append(_gc_r)
    _g_summ_rows.append({**_gs_r, 'run': _rn})

_gears_run_class_df = pd.concat(_g_cls_rows, ignore_index=True)
_ate_cols  = ['ate_all', 'ate_single', 'ate_combo', 'ate_held']
_gears_ate = pd.read_csv(GEARS_SUMM_CACHE)[['run'] + _ate_cols]
_gears_run_summ_df = pd.DataFrame(_g_summ_rows).merge(_gears_ate, on='run', how='left')
print(f'GEARS per-run metrics rebuilt with merged synergy (ate_* preserved).')


# ── CI-ready long dataframes used by panel E ─────────────────────────────
# SHERLOCK and GEARS all have N_RUNS rows per class — all get CI bars
class_df_ci = pd.concat(
    [_run_class_df, _gears_run_class_df], ignore_index=True,
)

# For summary metrics
_SUMM_KEYS = ['f1_weighted', 'ari', 'ami']
_SUMM_NAMES = {'f1_weighted': 'F1', 'ari': 'ARI', 'ami': 'AMI'}
_slk_run_long = _run_summ_df[['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_slk_run_long['model'] = 'SHERLOCK'
_gears_run_long = _gears_run_summ_df[['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_gears_run_long['model'] = 'GEARS'
summ_long_ci = pd.concat(
    [_slk_run_long[['model', 'metric', 'score']],
     _gears_run_long[['model', 'metric', 'score']]],
    ignore_index=True,
)
summ_long_ci['Metric'] = pd.Categorical(
    summ_long_ci['metric'].map(_SUMM_NAMES),
    categories=list(_SUMM_NAMES.values()), ordered=True,
)
print(_run_summ_df[['run', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

In [ ]:
# ── Synergy UMAP — 12-metric embedding for SHERLOCK and GEARS ────────────
from sklearn.preprocessing import StandardScaler
from umap import UMAP


def _fit_umap(df, metrics):
    avail  = [c for c in metrics if c in df.columns]
    X      = df[avail].values.astype(float)
    good   = np.all(np.isfinite(X), axis=1)
    work   = df[good].copy()
    X_s    = StandardScaler().fit_transform(X[good])
    reducer = UMAP(n_neighbors=10, min_dist=0.25, random_state=42)
    emb    = reducer.fit_transform(X_s)
    work['UMAP1'] = emb[:, 0]
    work['UMAP2'] = emb[:, 1]
    return work


# ── Pick the highest-weighted-F1 model per method for the UMAPs (fresh) ──
# The UMAPs (and the derived summary + Panel D) use the best GI *classifier*
# per method — not the best-ATE run or the last GEARS run. Recompute the chosen
# models' GI tables from source (SHERLOCK: reload checkpoint + get_gi; GEARS: raw
# per-run preds) so a stale syn_best.csv can never feed the figure.
_slk_best_run   = int(_run_summ_df.loc[_run_summ_df['f1_weighted'].idxmax(), 'run'])
_gears_best_run = int(_gears_run_summ_df.loc[_gears_run_summ_df['f1_weighted'].idxmax(), 'run'])
print(f'UMAP models by weighted-F1  —  SHERLOCK run {_slk_best_run}  |  GEARS run {_gears_best_run}')

_slk_best_model = slk.tl.load_results(checkpoints[_slk_best_run])
slk.tl.eval_single(_slk_best_model, data, obsm_key='z', uns_key='results', device=DEVICE)
_syn_bestf1 = _slk_best_model.get_gi(data, min_cells=5, mode='observed')
slk_classified, slk_class_df, slk_summary = evaluate_gi_classification(
    _syn_bestf1, 'SHERLOCK', gi_series)

_gears_bestf1_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_gears_best_run}.csv')
gears_classified, gears_class_df, gears_summary = evaluate_gi_classification(
    _gears_bestf1_raw, 'GEARS', gi_series)

syn_umap   = _fit_umap(slk_classified,   GI_METRICS)
gears_umap = _fit_umap(gears_classified, GI_METRICS)


def _cluster_metrics(umap_df, model_name, metrics):
    """Cluster quality metrics in the original metric space (not UMAP 2D)."""
    both  = umap_df[umap_df['gt_gi'].notna()].copy()
    avail = [c for c in metrics if c in both.columns]
    X_raw = both[avail].values.astype(float)
    finite = np.all(np.isfinite(X_raw), axis=1)
    both  = both[finite]
    X_s   = StandardScaler().fit_transform(X_raw[finite])
    y_gt  = both['gt_gi'].values.astype(str)
    y_pred = both['predicted_gi'].values.astype(str)
    if len(np.unique(y_gt)) > 1:
        _sil_s   = silhouette_samples(X_s, y_gt)
        _per_cls = pd.Series(_sil_s, index=y_gt).groupby(level=0).mean()
        sil      = float(_per_cls.mean())   # macro-average (equal weight per class)
    else:
        sil = float('nan')
    return {
        'model':      model_name,
        'silhouette': sil,
        'ari':        adjusted_rand_score(y_gt, y_pred),
        'nmi':        normalized_mutual_info_score(y_gt, y_pred),
    }


slk_clust   = _cluster_metrics(syn_umap,   'SHERLOCK', GI_METRICS)
gears_clust = _cluster_metrics(gears_umap, 'GEARS',    GI_METRICS)

for sd, cd in [(slk_summary, slk_clust), (gears_summary, gears_clust)]:
    sd['silhouette'] = cd['silhouette']
summary_df = pd.DataFrame([slk_summary, gears_summary])
print(summary_df[['model', 'f1_weighted', 'silhouette', 'ari', 'ami']].to_string(index=False))


# ── GI representative examples — used by panel D arrows & panel F heatmap ─
_cats_gi = ['approximately additive', 'epistasis', 'neomorphic',
            'potentiation', 'redundant', 'suppression', 'synergy']
_syn_all = {'synergy (similar phenotype)', 'synergy (dissimilar phenotype)', 'synergy'}
_val_set = set(val_combos)
_correct_gi = (
    (slk_classified['predicted_gi'] == slk_classified['gt_gi']) |
    (slk_classified['predicted_gi'].isin(_syn_all) & slk_classified['gt_gi'].isin(_syn_all))
)

_gi_examples = {}
_cbl_cnn1_idx = canon_pair('CBL+CNN1')
if _cbl_cnn1_idx in slk_classified.index:
    _gi_examples['synergy'] = slk_classified.loc[_cbl_cnn1_idx]
else:
    _p = slk_classified[slk_classified['gt_gi'].isin(_syn_all) &
                         slk_classified['predicted_gi'].isin(_syn_all)]
    _h = _p[_p.index.isin(_val_set)]
    _gi_examples['synergy'] = (_h if not _h.empty else _p).iloc[0]

for _cat in _cats_gi:
    if _cat == 'synergy':
        continue
    _p = slk_classified[_correct_gi & (slk_classified['gt_gi'] == _cat)]
    if _p.empty:
        continue
    _h = _p[_p.index.isin(_val_set)]
    _gi_examples[_cat] = (_h if not _h.empty else _p).iloc[0]

_gi_arrows = {}
for _cat, _row in _gi_examples.items():
    _na2, _nb2 = str(_row['pert_a']), str(_row['pert_b'])
    _cidx = canon_pair(f'{_na2}+{_nb2}')
    if _cidx in syn_umap.index:
        _gi_arrows[f'{_na2[:5]}/{_nb2[:5]}'] = _cidx

print(f'GI arrows ({len(_gi_arrows)}): {list(_gi_arrows.keys())}')

# ── Biologically motivated arrow annotations — GT-labelled pairs ──────────
HIGHLIGHT_COMBOS = {
    'DUSP9/MAPK1':   canon_pair('DUSP9+MAPK1'),
    'MAP2K6/SPI1':   canon_pair('MAP2K6+SPI1'),
    'CEBPA/B':       canon_pair('CEBPA+CEBPB'),
    'CBFA2T3/FEV':   canon_pair('FEV+CBFA2T3'),
    'CBL/PTPN12':    canon_pair('CBL+PTPN12'),
}

arrow_pairs = {}
for display, canonical in HIGHLIGHT_COMBOS.items():
    if canonical in syn_umap.index:
        row = syn_umap.loc[canonical]
        arrow_pairs[display] = canonical
        print(f'  GT  {display:18s} → GT: {row.get("gt_gi","N/A")}  Pred: {row.get("predicted_gi","N/A")}')
    else:
        print(f'  GT  {display:18s} not found in syn_umap')

# ── Novel annotations — unlabelled combinations of biological interest ────
NOVEL_COMBOS = {
    'BAK1/Bim':   canon_pair('BAK1+BCL2L11'),
    'p21/p27':    canon_pair('CDKN1A+CDKN1B'),
    'CEBPA/JUN':  canon_pair('CEBPA+JUN'),
    'UBASH3A/B':  canon_pair('UBASH3A+UBASH3B'),
}

novel_arrow_pairs = {}
for display, canonical in NOVEL_COMBOS.items():
    if canonical in syn_umap.index:
        row = syn_umap.loc[canonical]
        novel_arrow_pairs[display] = canonical
        print(f'  Novel {display:16s} → Pred: {row.get("predicted_gi","N/A")}')
    else:
        print(f'  Novel {display:16s} not found in syn_umap')

print(f'\n{len(arrow_pairs)} GT-labelled + {len(novel_arrow_pairs)} novel combinations annotated.')

In [ ]:
# ── GI-classification significance helper ────────────────────────────────
# SHERLOCK vs GEARS Welch t-test per summary metric (F1 / ARI / AMI), with a
# Benjamini-Hochberg FDR correction across the three metrics, drawn as a
# ggsignif-style bracket above each bar pair. Reused by every Panel-E summary.
def add_gi_significance(ax, long_df, model_a='SHERLOCK', model_b='GEARS',
                        metric_order=('F1', 'ARI', 'AMI'),
                        width=0.55, n_hue=2, y_pad=0.05, tick=0.018, fs=5):
    """`long_df` has columns model / Metric / score (one row per run). Bars are a
    seaborn barplot of x='Metric', hue='model' at the given `width`, so the two
    bar centres per metric sit at i +/- width/(2*n_hue)."""
    from scipy import stats as _st

    _p, _top = [], []
    for _m in metric_order:
        _a = long_df[(long_df['Metric'] == _m) & (long_df['model'] == model_a)]['score'].dropna().to_numpy()
        _b = long_df[(long_df['Metric'] == _m) & (long_df['model'] == model_b)]['score'].dropna().to_numpy()
        _p.append(_st.ttest_ind(_a, _b, equal_var=False).pvalue if len(_a) >= 2 and len(_b) >= 2 else np.nan)
        _cand = [np.nanmean(_a) + np.nanstd(_a) if len(_a) else np.nan,
                 np.nanmean(_b) + np.nanstd(_b) if len(_b) else np.nan]
        _top.append(np.nanmax(_cand) if np.any(np.isfinite(_cand)) else 1.0)

    # Benjamini-Hochberg across the three metrics
    _pa = np.array(_p, float); _adj = np.full_like(_pa, np.nan)
    _valid = np.where(np.isfinite(_pa))[0]
    _ordv = _valid[np.argsort(_pa[_valid])]; _mtot = len(_ordv); _prev = 1.0
    for _k in range(_mtot - 1, -1, -1):
        _ii = _ordv[_k]; _prev = min(_prev, _pa[_ii] * _mtot / (_k + 1)); _adj[_ii] = min(_prev, 1.0)

    def _stars(_q):
        if not np.isfinite(_q):
            return ''
        return '***' if _q < 1e-3 else '**' if _q < 1e-2 else '*' if _q < 0.05 else 'ns'

    # Asterisks sit high in the text box, 'ns' sits on the baseline; nudge each so
    # the ink lands at the same height above every bracket regardless of glyph.
    _lbl_dy = {'ns': 0.006}
    _x_a = -width / 2 + width / (2 * n_hue)                 # model_a bar offset within a group
    _x_b = _x_a + (n_hue - 1) * (width / n_hue)             # model_b bar offset
    _ytop = -np.inf
    for _i, _m in enumerate(metric_order):
        _lbl = _stars(_adj[_i])
        if not _lbl:
            continue
        _x1, _x2 = _i + _x_a, _i + _x_b
        _y = _top[_i] + y_pad
        ax.plot([_x1, _x1, _x2, _x2], [_y, _y + tick, _y + tick, _y],
                lw=0.5, c='0.35', clip_on=False)
        ax.text((_x1 + _x2) / 2, _y + tick + _lbl_dy.get(_lbl, -0.014), _lbl,
                ha='center', va='bottom', fontsize=fs, color='0.2', clip_on=False)
        _ytop = max(_ytop, _y + tick + 0.04)
    if np.isfinite(_ytop):
        _lo, _hi = ax.get_ylim()
        ax.set_ylim(_lo, max(_hi, _ytop))
    return ax


In [ ]:
# ── Panel A: Counterfactual ATE — in-distribution and OOD ────────────────
METRIC_LABELS = {
    'ate_all':    'ATE\n(all)',
    'ate_single': 'ATE\n(singles)',
    'ate_combo':  'ATE\n(combos)',
    'ate_held':   'ATE\n(held-out combos)',
}

METHOD_ORDER = ['SHERLOCK', 'GEARS', 'cVAE', 'sVAE', 'contrastiveVI']
METHOD_PALETTE = {
    'SHERLOCK':      CLR_SHERLOCK,
    'cVAE':          '#33a02c',
    'sVAE':          '#ff7f00',
    'contrastiveVI': '#6a3d9a',
    'GEARS':         CLR_GEARS,
}

sherlock_ate_df = runs_df[['run'] + list(METRIC_LABELS)].copy()
sherlock_ate_df['method'] = 'SHERLOCK'
gears_ate_df = _gears_run_summ_df[['run'] + list(METRIC_LABELS)].copy()
gears_ate_df['method'] = 'GEARS'
all_ate_df = pd.concat(
    [sherlock_ate_df, gears_ate_df,
     bench_runs_df[['method', 'run'] + list(METRIC_LABELS)]],
    ignore_index=True,
)

df_long = all_ate_df.melt(
    id_vars=['method', 'run'],
    value_vars=list(METRIC_LABELS),
    var_name='metric', value_name='Pearson r',
)
df_long['Metric'] = pd.Categorical(
    df_long['metric'].map(METRIC_LABELS),
    categories=[METRIC_LABELS[k] for k in METRIC_LABELS],
    ordered=True,
)

TEXT_SCALE_A = -2.  # +/- pt added to all Panel A text uniformly; 0.0 == RC defaults
TEXT_SCALE_B = -1.5

fig_a, ax_a = plt.subplots(figsize=(300/PPI, 150/PPI), constrained_layout=True)

sns.boxplot(
    data=df_long, x='Metric', y='Pearson r', hue='method',
    hue_order=METHOD_ORDER, palette=METHOD_PALETTE,
    width=0.75, linewidth=0.6, fliersize=0, ax=ax_a,
    boxprops=dict(alpha=0.55),
)
sns.stripplot(
    data=df_long, x='Metric', y='Pearson r', hue='method',
    hue_order=METHOD_ORDER, palette=METHOD_PALETTE,
    dodge=True, size=2.6 + TEXT_SCALE_A/3, jitter=0.1,
    linewidth=0.3, edgecolor='white', ax=ax_a, legend=False,
)

ax_a.axvline(2.5, color='0.55', linewidth=0.8, linestyle='--', alpha=0.7)
ax_a.set_ylim(bottom=0)

# ── Significance: SHERLOCK vs every other method, per column ──────────────
# Welch's t-test on the per-run values, Benjamini-Hochberg FDR across all
# comparisons; drawn as R/ggsignif-style brackets with ns / * / ** / ***.
from scipy import stats as _stats
_sig_metrics = list(METRIC_LABELS)                 # ate_all, ate_single, ate_combo, ate_held
_sig_others  = [m for m in METHOD_ORDER if m != 'SHERLOCK']

def _mvals(_mk, _meth):
    return df_long[(df_long['metric'] == _mk) &
                   (df_long['method'] == _meth)]['Pearson r'].dropna().to_numpy()

_comps = []                                        # [col_i, other_method, p]
for _i, _mk in enumerate(_sig_metrics):
    _s = _mvals(_mk, 'SHERLOCK')
    for _o in _sig_others:
        _ov = _mvals(_mk, _o)
        if len(_s) < 2 or len(_ov) < 2:
            continue
        _comps.append([_i, _o, _stats.ttest_ind(_s, _ov, equal_var=False).pvalue])

# Benjamini-Hochberg across the whole family
_pv = np.array([c[2] for c in _comps])
_ord = np.argsort(_pv); _mtot = len(_pv); _adj = np.empty(_mtot); _prev = 1.0
for _k in range(_mtot - 1, -1, -1):
    _ix = _ord[_k]; _prev = min(_prev, _pv[_ix] * _mtot / (_k + 1)); _adj[_ix] = min(_prev, 1.0)
for _c, _pa in zip(_comps, _adj):
    _c.append(_pa)

def _stars(_p):
    return '***' if _p < 1e-3 else '**' if _p < 1e-2 else '*' if _p < 0.05 else 'ns'

# dodge geometry (boxplot width=0.75, one box per method)
_n_hue, _box_w = len(METHOD_ORDER), 0.75
def _xpos(_i, _meth):
    _j = METHOD_ORDER.index(_meth)
    return _i - _box_w / 2 + _box_w / (2 * _n_hue) + _j * (_box_w / _n_hue)

_y0   = float(df_long['Pearson r'].max()) + 0.03   # bracket base, above the boxes
_step = 0.052                                      # gap between stacked brackets
_tick = 0.012                                      # bracket end-tick height
_sfs  = 6 + TEXT_SCALE_A
_ytop = _y0
for _i in range(len(_sig_metrics)):
    _col = sorted([c for c in _comps if c[0] == _i],
                  key=lambda c: METHOD_ORDER.index(c[1]))   # nearest (GEARS) lowest
    for _lvl, (_, _o, _p, _pa) in enumerate(_col):
        _x1, _x2 = _xpos(_i, 'SHERLOCK'), _xpos(_i, _o)
        _y = _y0 + _lvl * _step
        ax_a.plot([_x1, _x1, _x2, _x2], [_y, _y + _tick, _y + _tick, _y],
                  lw=0.5, c='0.35', clip_on=False)
        ax_a.text((_x1 + _x2) / 2, _y + _tick, _stars(_pa),
                  ha='center', va='bottom', fontsize=_sfs, color='0.2', clip_on=False)
        _ytop = max(_ytop, _y + _tick)
ax_a.set_ylim(0, _ytop + 0.05)
_ylim = ax_a.get_ylim()
_y_ann = _ylim[1] + 0.07 * (_ylim[1] - _ylim[0])
ax_a.text(1.0, _y_ann, 'in distribution', ha='center', va='top',
          fontsize=5.5 , color='0.45', style='italic')
ax_a.text(3.0, _y_ann, 'out of distribution', ha='center', va='top',
          fontsize=5.5 , color='0.45', style='italic')

ax_a.set_xlabel('')
ax_a.set_ylabel('Pearson $r$', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_B)
ax_a.set_title('Counterfactual ATE', pad=9, fontsize=mpl.rcParams['axes.titlesize'] )
ax_a.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
ax_a.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_B)
ax_a.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_B)
ax_a.text(0.02, 0.04, f'$n$ = {N_RUNS} runs',
          transform=ax_a.transAxes, ha='left', va='bottom',
          fontsize=6 + TEXT_SCALE_A, color='0.4')
ax_a.legend(
    title='Method', frameon=False, loc='upper right',
    bbox_to_anchor=(1.15, 0.95),
    fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_A,
    title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_A,
)

fig_a.savefig(OUT_DIR / 'panel_a_ate.svg')
plt.show()
print('Panel A saved.')

In [ ]:
# ── Panel B: reproducibility across runs + accuracy against the data ─────
# Four columns, two kinds of measure:
#   cross-run (leave-one-out Spearman: one run vs the mean of the others)
#     * perturbation correlation matrix
#     * gate co-activation matrix
#     * counterfactual effect profiles (perturbation x gene)
#   accuracy (within run, Pearson vs the measured effects)
#     * counterfactual effect sizes (per-perturbation effect magnitudes)

# Per-run counterfactual effect matrices, cached alongside the other matrices.
CF_PATH = OUT_DIR / 'sherlock_cf_effects.npz'
if CF_PATH.exists():
    _cf = np.load(CF_PATH, allow_pickle=True)
else:
    _store, _obs = {}, None
    for _rn in range(1, N_RUNS + 1):
        _m = slk.tl.load_results(checkpoints[_rn]).to(DEVICE)
        _ate = _m.predict_counterfactual_effects(
            data, n_centroids=25, observed_effect=data.uns['treat_effect'], approximate=False,
        )
        _p, _o = _ate['predicted_df'], _ate['observed_df']
        _common = _p.index.intersection(_o.index)
        _store[f'pred_{_rn}'] = _p.loc[_common].values.astype(np.float32)
        if _obs is None:
            _obs = _o.loc[_common].values.astype(np.float32)
            _store['perts'] = np.array(_common, dtype=object)
        del _m
    _store['obs'] = _obs
    np.savez_compressed(CF_PATH, **_store)
    _cf = np.load(CF_PATH, allow_pickle=True)

pred_by_run = {int(k.split('_')[-1]): _cf[k] for k in _cf.files if k.startswith('pred_')}
obs_mat     = _cf['obs']


def _loo_spearman_flat(mats_by_run, metric_name):
    """Leave-one-out Spearman for non-symmetric (perturbation x gene) matrices."""
    rows, keys = [], sorted(mats_by_run)
    for k in keys:
        mean_other = np.mean([mats_by_run[j] for j in keys if j != k], axis=0)
        a, b = mats_by_run[k].ravel(), mean_other.ravel()
        v = np.isfinite(a) & np.isfinite(b)
        rows.append({'metric': metric_name, 'run': k, 'r': float(spearmanr(a[v], b[v])[0])})
    return pd.DataFrame(rows)


L_RHO, L_GATE = 'Perturbation\ncorrelation', 'Gate\nco-activation'
L_CFP, L_CFS  = 'Counterfactual\neffect profile', 'Counterfactual\neffect size'
ROBUST_LABELS = {'rho_corr_r': L_RHO, 'w_coact_r': L_GATE,
                 'cf_effect_r': L_CFP, 'effect_size_r': L_CFS}

_cf_cross = _loo_spearman_flat(pred_by_run, 'cf_effect_r')

df_robust = pd.concat([
    pairwise_df.rename(columns={'r': 'Score'})[['metric', 'Score']],
    _cf_cross.rename(columns={'r': 'Score'})[['metric', 'Score']],
    runs_df[['effect_size_r']].rename(columns={'effect_size_r': 'Score'}).assign(metric='effect_size_r'),
], ignore_index=True)
df_robust['Metric'] = pd.Categorical(
    df_robust['metric'].map(ROBUST_LABELS),
    categories=[ROBUST_LABELS[k] for k in ROBUST_LABELS], ordered=True,
)

print(df_robust.groupby('metric', observed=True)['Score'].agg(['mean', 'std']).round(3).to_string())

CLR_ACC = '#7f7f7f'
_pal = {L_RHO: CLR_SHERLOCK, L_GATE: CLR_SHERLOCK, L_CFP: CLR_SHERLOCK, L_CFS: CLR_ACC}

fig_b2, ax_b2 = plt.subplots(figsize=(4.4, 3.0))
sns.stripplot(data=df_robust, x='Metric', y='Score', hue='Metric', palette=_pal, legend=False,
              size=5, jitter=0.15, linewidth=0.4, edgecolor='white', ax=ax_b2)
sns.boxplot(data=df_robust, x='Metric', y='Score', color='white', width=0.45, linewidth=0.75,
            fliersize=0, ax=ax_b2, boxprops=dict(edgecolor='0.3'),
            whiskerprops=dict(color='0.3'), capprops=dict(color='0.3'),
            medianprops=dict(color='0.15', linewidth=1.5))
ax_b2.set_ylim(0, 1.18)
ax_b2.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax_b2.set_xlabel('')
ax_b2.set_ylabel('Correlation')
ax_b2.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
ax_b2.axvline(2.5, color='0.75', lw=0.6, ls='--')
ax_b2.plot([-0.4, 2.4], [1.12, 1.12], color=CLR_SHERLOCK, lw=1, clip_on=False)
ax_b2.text(1.0, 1.14, 'reproducibility across runs', ha='center', va='bottom',
           fontsize=7, color=CLR_SHERLOCK)
ax_b2.plot([2.6, 3.4], [1.12, 1.12], color=CLR_ACC, lw=1, clip_on=False)
ax_b2.text(3.0, 1.14, 'accuracy vs data', ha='center', va='bottom', fontsize=7, color=CLR_ACC)
ax_b2.text(0.99, 0.02, f'$n$ = {N_RUNS} runs', transform=ax_b2.transAxes,
           ha='right', va='bottom', fontsize=6, color='0.4')

plt.tight_layout()
fig_b2.savefig(OUT_DIR / 'panel_b_robustness.svg', bbox_inches='tight')
fig_b2.savefig(OUT_DIR / 'panel_b_robustness.png', dpi=300, bbox_inches='tight')
plt.show()
print('Panel B (robustness) saved.')


In [ ]:
slk_classified[slk_classified.index.str.contains('SET')]

In [ ]:
# ── Panel C: SHERLOCK latent space — Excel-associated perturbations ───────
# Three UMAPs of the SAME subset (all labeled "SHERLOCK latent space"):
#   1. coloured by perturbation name
#   2. coloured by group/cluster (wide figure so the legend fits)
#   3. coloured by single vs combination
# Genes + groups are hard-coded from aax4438_tables6.xlsx (both sheets combined;
# any gene appearing in >1 cluster was dropped, then clusters left with <2 genes
# were dropped). Subset = singles that are group genes + any combo with >=1 group
# gene (the partner need not be a group gene). One fresh UMAP is fit on the subset
# (no cache file, so nothing can go stale).
#
# Group label (fig 2 only): single-group clusters are shown; a combo whose two
# genes fall in DIFFERENT groups ("cluster / cluster") is dropped; a combo with
# only one gene in a group collapses into that gene's named group.
GENE_GROUPS = {
    'MAP2K':                   ['MAP2K3', 'MAP2K6'],
    'Ig domain containing':    ['IGDCC3', 'PRTG'],
    'Tyrosine phosphatase':    ['PTPN12', 'PTPN9'],
    'MAP4K':                   ['MAP4K3', 'MAP4K5'],
    'Brachyury TF':            ['CSRNP1', 'SET', 'TBX2', 'TBX3'],
    'Kinesin':                 ['KIF18B', 'KIF2C'],
    'bLZ TF':                  ['CEBPA', 'CEBPB', 'CEBPE', 'CELF2', 'FOSB', 'OSR2'],
    'Regulation of apoptosis': ['EGR1', 'FOXO4', 'IER5L', 'IKZF3', 'MEIS1', 'PRDM1',
                                'RHOXF2B', 'SAMD1', 'SGK1', 'TP73'],
    'Homeobox TF':             ['CBFA2T3', 'CITED1', 'DLX2', 'MIDN', 'NIT1', 'SNAI1', 'TMSB4X'],
}
_gene2group = {g: grp for grp, genes in GENE_GROUPS.items() for g in genes}
_EXCEL_GENES = set(_gene2group)

def _in_subset(name):
    """Single that is a group gene, or a combo with >=1 group gene."""
    return any(p in _EXCEL_GENES for p in str(name).split('+'))

def _group_of(name):
    """Group label: a single group; 'group A / group B' when the two genes
    fall in different groups; or None (no group gene). A combo with only one
    gene in a group collapses to that group."""
    parts = str(name).split('+')
    if len(parts) == 1:
        return _gene2group.get(parts[0])
    ga, gb = _gene2group.get(parts[0]), _gene2group.get(parts[1])
    if ga and gb:
        return ga if ga == gb else ' / '.join(sorted([ga, gb]))  # two groups
    return ga or gb                              # one gene in a group -> its group

# ── Build subset from the SHERLOCK latent space (same 'z' as the full panel) ──
_c_mask = all_sub.obs[p_key].astype(str).map(_in_subset).values
c_sub   = all_sub[_c_mask].copy()
c_sub.obs['pert_name']   = c_sub.obs[p_key].astype(str).values
c_sub.obs['group_label'] = c_sub.obs['pert_name'].map(_group_of).values
c_sub.obs['pert_type']   = np.where(
    c_sub.obs['pert_name'].str.contains(r'\+', regex=True), 'combination', 'single')

sc.pp.neighbors(c_sub, n_neighbors=15, use_rep='z')
sc.tl.umap(c_sub, random_state=42)
_coords = c_sub.obsm['X_umap']
print(f'Panel C subset: {c_sub.n_obs} cells | '
      f"{c_sub.obs['pert_name'].nunique()} perturbations | "
      f"{c_sub.obs['group_label'].nunique()} group categories (shown)")

def _palette(cats):
    """Distinct qualitative colours (tab20 + tab20b + tab20c = 60), cycled."""
    cats = list(cats)
    base = (list(plt.get_cmap('tab20').colors)
            + list(plt.get_cmap('tab20b').colors)
            + list(plt.get_cmap('tab20c').colors))
    return {c: base[i % len(base)] for i, c in enumerate(cats)}

def _style_umap(ax, coords, color_vals, palette, title):
    ax.scatter(coords[:, 0], coords[:, 1],
               c=[palette[v] for v in color_vals],
               s=0.8, alpha=0.4, linewidths=0, rasterized=True)
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_title(title, pad=4)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    _add_umap_axis_indicator(ax, length=1.0)

_TITLE = 'SHERLOCK latent space'

# ── 1. coloured by perturbation name (many perturbations -> no legend) ────
_name_cats = sorted(c_sub.obs['pert_name'].unique())
_name_pal  = _palette(_name_cats)
fig_c1, ax_c1 = plt.subplots(figsize=(196/PPI, 171/PPI), constrained_layout=True)
_style_umap(ax_c1, _coords, c_sub.obs['pert_name'].values, _name_pal, _TITLE)
fig_c1.savefig(OUT_DIR / 'panel_c_umap_byname.svg')
plt.show()

# ── 2. coloured by group/cluster (incl. two-group combos), subsampled ─────
SUBSAMPLE_FRAC = 0.3  # fraction of cells drawn in this panel (density control)
TEXT_SCALE_C   = -1.0  # +/- pt on the cluster legend text (shrink to fit the figsize)
_grp_valid = c_sub.obs['group_label'].notna().values
_grp_vals  = c_sub.obs['group_label'].values
_grp_cats  = sorted(set(_grp_vals[_grp_valid]))
_grp_pal   = _palette(_grp_cats)
# subsample cells so the embedding reads less dense (legend keeps all cats)
_grp_plot  = _grp_valid & (np.random.default_rng(0).random(c_sub.n_obs) < SUBSAMPLE_FRAC)
fig_c2 = plt.figure(figsize=(285/PPI, 185/PPI))
_gs_c2  = fig_c2.add_gridspec(1, 2, width_ratios=[2.5, 1.0], wspace=0.08)
ax_c2   = fig_c2.add_subplot(_gs_c2[0, 0])
_ax_leg = fig_c2.add_subplot(_gs_c2[0, 1]); _ax_leg.axis('off')
_style_umap(ax_c2, _coords[_grp_plot], _grp_vals[_grp_plot], _grp_pal, _TITLE)
_ax_leg.legend(
    handles=[Patch(color=_grp_pal[g], label=g.replace(' / ', ' /\n')) for g in _grp_cats],
    loc='center left', frameon=False, fontsize=5 + TEXT_SCALE_C,
    handlelength=0.9, handletextpad=0.4, labelspacing=0.4, borderaxespad=0,
    title='Cluster', title_fontsize=6 + TEXT_SCALE_C,
)
fig_c2.savefig(OUT_DIR / 'panel_c_umap_bygroup.svg')
plt.show()

# ── 3. coloured by single vs combination ─────────────────────────────────
_type_pal = {'single': CLR_SHERLOCK, 'combination': '#E8735A'}
fig_c3, ax_c3 = plt.subplots(figsize=(196/PPI, 171/PPI), constrained_layout=True)
_style_umap(ax_c3, _coords, c_sub.obs['pert_type'].values, _type_pal, _TITLE)
ax_c3.legend(
    handles=[Patch(color=v, label=k.capitalize()) for k, v in _type_pal.items()],
    frameon=False, loc='lower right', handlelength=0.8, borderaxespad=0,
)
fig_c3.savefig(OUT_DIR / 'panel_c_umap_bytype.svg')
plt.show()

print('Panel C (3 UMAPs) saved.')

In [ ]:
# ── SHERLOCK latent space — one figure per double (two-group) cluster ─────
# For each "A / B" dual grouping, colour its combos plus the two single-group
# clusters A and B; everything else is grey. Uses the single shared manifold
# (_coords) built in the Panel C cell above.
_dual_labels = sorted(l for l in c_sub.obs['group_label'].dropna().unique()
                      if ' / ' in str(l))
print(f'{len(_dual_labels)} double groupings: {_dual_labels}')

for _dual in _dual_labels:
    _a, _b  = _dual.split(' / ')
    _cats   = [_a, _b, _dual]
    _pal    = _palette(_cats)               # A, B, A/B -> three distinct colours
    _gl     = c_sub.obs['group_label'].values
    _mask   = np.isin(_gl, _cats)

    fig_d  = plt.figure(figsize=(420/PPI, 185/PPI))
    _gs_d   = fig_d.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.02)
    ax_d    = fig_d.add_subplot(_gs_d[0, 0])
    _ax_dl  = fig_d.add_subplot(_gs_d[0, 1]); _ax_dl.axis('off')

    # grey background (everything not in A, B, or A/B)
    ax_d.scatter(_coords[~_mask, 0], _coords[~_mask, 1],
                 c='0.85', s=0.8, alpha=0.4, linewidths=0, rasterized=True)
    # A, B and the A/B combos on top
    ax_d.scatter(_coords[_mask, 0], _coords[_mask, 1],
                 c=[_pal[v] for v in _gl[_mask]],
                 s=3.0, alpha=0.9, linewidths=0, rasterized=True)

    ax_d.set_xlabel(''); ax_d.set_ylabel('')
    ax_d.set_title('SHERLOCK latent space', pad=4)
    ax_d.set_xticks([]); ax_d.set_yticks([])
    for sp in ax_d.spines.values():
        sp.set_visible(False)
    _add_umap_axis_indicator(ax_d, length=1.0)

    _ax_dl.legend(
        handles=[Patch(color=_pal[c], label=c) for c in _cats],
        loc='center left', frameon=False, fontsize=5,
        handlelength=0.9, handletextpad=0.4, labelspacing=0.3, borderaxespad=0,
        title=_dual, title_fontsize=6,
    )
    _svg = _dual.replace(' / ', '__').replace(' ', '_')
    fig_d.savefig(OUT_DIR / f'panel_c_umap_dual_{_svg}.svg', bbox_inches='tight')
    plt.show()

print('Double-group UMAPs saved.')

In [ ]:
gi_labels[gi_labels['combination'].str.contains('PTPN12')]

In [ ]:
# ── SHERLOCK latent space — one combo + its two singles ──────────────────
# Colours a chosen combo and its two constituent single perturbations by name;
# everything else grey. Uses the shared manifold (_coords) from the Panel C cell.
HIGHLIGHT_COMBO = 'IKZF3+MAP2K6'

_ca, _cb = HIGHLIGHT_COMBO.split('+')
_pn = c_sub.obs['pert_name'].astype(str)
_is_combo = _pn.map(lambda n: '+' in n and sorted(n.split('+')) == sorted([_ca, _cb])).values

_lab = pd.Series(index=c_sub.obs.index, dtype=object)
_lab[(_pn == _ca).values] = _ca
_lab[(_pn == _cb).values] = _cb
_lab[_is_combo]           = HIGHLIGHT_COMBO
_show = _lab.notna().values

_cats = [_ca, _cb, HIGHLIGHT_COMBO]
_pal  = _palette(_cats)

fig_hc = plt.figure(figsize=(420/PPI, 185/PPI))
_gs_hc  = fig_hc.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.02)
ax_hc   = fig_hc.add_subplot(_gs_hc[0, 0])
_ax_hcl = fig_hc.add_subplot(_gs_hc[0, 1]); _ax_hcl.axis('off')

# grey background (everything not in the combo or its singles)
ax_hc.scatter(_coords[~_show, 0], _coords[~_show, 1],
              c='0.85', s=0.8, alpha=0.4, linewidths=0, rasterized=True)
# the combo + its two singles on top
ax_hc.scatter(_coords[_show, 0], _coords[_show, 1],
              c=[_pal[l] for l in _lab.values[_show]],
              s=3.0, alpha=0.9, linewidths=0, rasterized=True)

ax_hc.set_xlabel(''); ax_hc.set_ylabel('')
ax_hc.set_title('SHERLOCK latent space', pad=4)
ax_hc.set_xticks([]); ax_hc.set_yticks([])
for sp in ax_hc.spines.values():
    sp.set_visible(False)
_add_umap_axis_indicator(ax_hc, length=1.0)

_ax_hcl.legend(
    handles=[Patch(color=_pal[c], label=c) for c in _cats],
    loc='center left', frameon=False, fontsize=5,
    handlelength=0.9, handletextpad=0.4, labelspacing=0.3, borderaxespad=0,
    title=HIGHLIGHT_COMBO, title_fontsize=6,
)
_svg = HIGHLIGHT_COMBO.replace('+', '_')
fig_hc.savefig(OUT_DIR / f'panel_c_umap_combo_{_svg}.svg', bbox_inches='tight')
plt.show()
print(f'{HIGHLIGHT_COMBO}: {int(_is_combo.sum())} combo cells | '
      f'{int((_pn == _ca).sum())} {_ca} | {int((_pn == _cb).sum())} {_cb}')

In [ ]:
# ── SHERLOCK latent space — one combo + its two singles ──────────────────
# Colours a chosen combo and its two constituent single perturbations by name;
# everything else grey. Uses the shared manifold (_coords) from the Panel C cell.
HIGHLIGHT_COMBO = 'CEBPE+SET'

_ca, _cb = HIGHLIGHT_COMBO.split('+')
_pn = c_sub.obs['pert_name'].astype(str)
_is_combo = _pn.map(lambda n: '+' in n and sorted(n.split('+')) == sorted([_ca, _cb])).values

_lab = pd.Series(index=c_sub.obs.index, dtype=object)
_lab[(_pn == _ca).values] = _ca
_lab[(_pn == _cb).values] = _cb
_lab[_is_combo]           = HIGHLIGHT_COMBO
_show = _lab.notna().values

_cats = [_ca, _cb, HIGHLIGHT_COMBO]
_pal  = _palette(_cats)

fig_hc = plt.figure(figsize=(420/PPI, 185/PPI))
_gs_hc  = fig_hc.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.02)
ax_hc   = fig_hc.add_subplot(_gs_hc[0, 0])
_ax_hcl = fig_hc.add_subplot(_gs_hc[0, 1]); _ax_hcl.axis('off')

# grey background (everything not in the combo or its singles)
ax_hc.scatter(_coords[~_show, 0], _coords[~_show, 1],
              c='0.85', s=0.8, alpha=0.4, linewidths=0, rasterized=True)
# the combo + its two singles on top
ax_hc.scatter(_coords[_show, 0], _coords[_show, 1],
              c=[_pal[l] for l in _lab.values[_show]],
              s=3.0, alpha=0.9, linewidths=0, rasterized=True)

ax_hc.set_xlabel(''); ax_hc.set_ylabel('')
ax_hc.set_title('SHERLOCK latent space', pad=4)
ax_hc.set_xticks([]); ax_hc.set_yticks([])
for sp in ax_hc.spines.values():
    sp.set_visible(False)
_add_umap_axis_indicator(ax_hc, length=1.0)

_ax_hcl.legend(
    handles=[Patch(color=_pal[c], label=c) for c in _cats],
    loc='center left', frameon=False, fontsize=5,
    handlelength=0.9, handletextpad=0.4, labelspacing=0.3, borderaxespad=0,
    title=HIGHLIGHT_COMBO, title_fontsize=6,
)
_svg = HIGHLIGHT_COMBO.replace('+', '_')
fig_hc.savefig(OUT_DIR / f'panel_c_umap_combo_{_svg}.svg', bbox_inches='tight')
plt.show()
print(f'{HIGHLIGHT_COMBO}: {int(_is_combo.sum())} combo cells | '
      f'{int((_pn == _ca).sum())} {_ca} | {int((_pn == _cb).sum())} {_cb}')

In [ ]:
# ── Panel D: Synergy UMAP — GT class (left) vs predicted class (right) ──
# Display-only relabelling: the two synergy sub-classes are shown as one
# "synergy" and "approximately additive" is shown as "additive". Underlying
# gt_gi / predicted_gi columns (and all metrics) are unchanged.
_GI_DISPLAY = {
    'synergy (similar phenotype)':    'synergy',
    'synergy (dissimilar phenotype)': 'synergy',
    'approximately additive':         'additive',
}
PALETTE_GI_DISPLAY = {
    'additive':     '#4DAF4A',  # green
    'epistasis':    '#FF7F00',  # orange
    'neomorphic':   '#984EA3',  # purple
    'potentiation': '#F781BF',  # pink
    'redundant':    '#A65628',  # brown
    'suppression':  '#E41A1C',  # red
    'synergy':      '#377EB8',  # blue
}

def _draw_synergy_umap(ax, umap_df, colour_col, title,
                       val_combos=None, arrow_pairs=None, show_unlabelled=False,
                       right_labels=None, show_axis=True, text_scale=0.0, margin=0.1):
    """
    colour_col: column to colour by (gt_gi or predicted_gi).
    show_unlabelled: if True, draw NaN rows as gray (for GT panel only).
    arrow_pairs: {display_label: canonical_idx}
    """
    labelled   = umap_df[umap_df[colour_col].notna()].copy()
    unlabelled = umap_df[umap_df[colour_col].isna()].copy()
    # merge synergy sub-classes -> 'synergy', 'approximately additive' -> 'additive'
    labelled[colour_col] = labelled[colour_col].map(lambda _s: _GI_DISPLAY.get(str(_s), _s))

    if show_unlabelled and len(unlabelled) > 0:
        ax.scatter(
            unlabelled['UMAP1'], unlabelled['UMAP2'],
            color='0.75', s=10, alpha=0.45, linewidths=0, zorder=1,
        )

    if val_combos is not None:
        in_dist  = labelled[~labelled.index.isin(val_combos)]
        out_dist = labelled[labelled.index.isin(val_combos)]
    else:
        in_dist  = labelled
        out_dist = labelled.iloc[0:0]

    for cls in sorted(labelled[colour_col].unique()):
        color   = PALETTE_GI_DISPLAY.get(str(cls), '#888888')
        grp_id  = in_dist[in_dist[colour_col] == cls]
        grp_ood = out_dist[out_dist[colour_col] == cls]
        if len(grp_id) > 0:
            ax.scatter(
                grp_id['UMAP1'], grp_id['UMAP2'],
                color=color, marker='o', s=18, alpha=0.85,
                linewidths=0.3, edgecolors='white', zorder=2,
            )
        if len(grp_ood) > 0:
            ax.scatter(
                grp_ood['UMAP1'], grp_ood['UMAP2'],
                color=color, marker='^', s=22, alpha=0.85,
                linewidths=0.3, edgecolors='white', zorder=3,
            )

    if arrow_pairs:
        for display_label, idx in arrow_pairs.items():
            if idx not in umap_df.index:
                continue
            row  = umap_df.loc[idx]
            x, y = float(row['UMAP1']), float(row['UMAP2'])
            _right = right_labels and display_label in right_labels
            ax.annotate(
                display_label,
                xy=(x, y), xytext=(x + 1.5 if _right else x - 1.5, y + 0.8),
                fontsize=6.5 + text_scale, color='0.1',
                ha='left' if _right else 'right', va='bottom',
                arrowprops=dict(
                    arrowstyle='->', color='0.35',
                    lw=0.8, shrinkA=2, shrinkB=2,
                ),
            )

    ax.margins(margin)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, pad=3, y=1.1, fontsize=mpl.rcParams['axes.titlesize'] + text_scale)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    if show_axis:
        _add_umap_axis_indicator(ax, length=1.0, fontsize=mpl.rcParams['axes.labelsize'] + text_scale)


TEXT_SCALE = - 1.0  # +/- pt added to all Panel D text uniformly; 0.0 == RC defaults

fig_d, axes_d = plt.subplots(1, 2, figsize=(312/PPI, 172/PPI), constrained_layout=True)

_draw_synergy_umap(axes_d[0], syn_umap, 'gt_gi', 'Norman GI Labels',
                   val_combos=val_combos, arrow_pairs=None,
                   show_unlabelled=True, show_axis=True, text_scale=TEXT_SCALE, margin=0.1)
_draw_synergy_umap(axes_d[1], syn_umap, 'predicted_gi', 'Sherlock GI Prediction',
                   val_combos=val_combos, arrow_pairs=_gi_arrows,
                   show_unlabelled=False, show_axis=True, text_scale=TEXT_SCALE, margin=0.1,
                   right_labels={'CBL/CNN1', 'CEBPB/PTPN1'})

# Legend
SHOW_LEGEND = True  # temporarily off to check panel sizing on the 371x172px canvas
class_handles = [Patch(color=PALETTE_GI_DISPLAY.get(cls, '#888888'), label=cls)
                 for cls in sorted(PALETTE_GI_DISPLAY)]
extra_handles = [
    Patch(color='0.75', label='No GT label'),
    Patch(color='none', label=''),
    Line2D([0], [0], marker='o', color='0.4', linestyle='None', ms=5, label='In-distribution'),
    Line2D([0], [0], marker='^', color='0.4', linestyle='None', ms=5, label='Held-out (OOD)'),
]
if SHOW_LEGEND:
    axes_d[1].legend(
        handles=class_handles + extra_handles,
        title='GI class', frameon=False,
        fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE,
        title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE,
        bbox_to_anchor=(1.15, 1), loc='upper left',
        handlelength=0.8, handleheight=0.8,
    )

fig_d.savefig(OUT_DIR / 'panel_d_synergy_umap.svg')
plt.show()
print('Panel D (synergy UMAP) saved.')

In [ ]:
# ── Panel D (GEARS): Synergy UMAP — GT class (left) vs GEARS predicted class (right) ──
fig_d_gears, axes_d_gears = plt.subplots(1, 2, figsize=(312/PPI, 172/PPI), constrained_layout=True)

_draw_synergy_umap(axes_d_gears[0], gears_umap, 'gt_gi', 'Norman GI Labels',
                   val_combos=val_combos, arrow_pairs=None,
                   show_unlabelled=True, show_axis=True, text_scale=TEXT_SCALE, margin=0.1)
_draw_synergy_umap(axes_d_gears[1], gears_umap, 'predicted_gi', 'GEARS GI Prediction',
                   val_combos=val_combos, #arrow_pairs=_gi_arrows,
                   show_unlabelled=False, show_axis=True, text_scale=TEXT_SCALE, margin=0.1,
                   right_labels={'CBL/CNN1', 'CEBPB/PTPN1'})

if SHOW_LEGEND:
    axes_d_gears[1].legend(
        handles=class_handles + extra_handles,
        title='GI class', frameon=False,
        fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE,
        title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE,
        bbox_to_anchor=(1.15, 1), loc='upper left',
        handlelength=0.8, handleheight=0.8,
    )

fig_d_gears.savefig(OUT_DIR / 'panel_d_synergy_umap_gears.svg')
plt.show()
print('Panel D (GEARS synergy UMAP) saved.')


In [ ]:
# ── Panel E: GI classification — per-class AUC (left) + summary (right) ──
MODEL_ORDER = ['SHERLOCK', 'GEARS']
class_order = (
    class_df_ci.groupby('class')['auc_roc']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bar_pal = {'SHERLOCK': CLR_SHERLOCK, 'GEARS': CLR_GEARS}

TEXT_SCALE_E = -1.0  # +/- pt added to all Panel E text uniformly; 0.0 == RC defaults

fig_e, (ax_e1, ax_e2) = plt.subplots(
    1, 2, figsize=(300/PPI, 150/PPI), constrained_layout=True,
    gridspec_kw={'width_ratios': [2.2, 1.8]},
)

# Left: per-class AUC-ROC
sns.barplot(
    data=class_df_ci.dropna(subset=['auc_roc']),
    y='class', x='auc_roc',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, order=class_order,
    width=0.62, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_e1, legend=False,
)
ax_e1.axvline(0.5, color='0.4', linewidth=0.4, linestyle='--')
ax_e1.set_xlabel('AUC-ROC (one-vs-rest)', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E - 1)
ax_e1.set_ylabel('')
_CLASS_LABEL_MAP = {
    'approximately additive': 'additive',
    # syn-sim / syn-dis are merged into a single 'synergy' class upstream
    # (MERGE_SYNERGY in the evaluate_gi_classification cell).
}
ax_e1.set_yticklabels([_CLASS_LABEL_MAP.get(c, c) for c in class_order])
ax_e1.set_xlim(0, 1)
ax_e1.set_xticks([0, 0.5, 1.0])
ax_e1.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_e1.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_e1.spines['left'].set_linewidth(0.4)
ax_e1.spines['bottom'].set_linewidth(0.4)
ax_e1.tick_params(axis='both', width=0.4)
ax_e1.spines['top'].set_visible(False)
ax_e1.spines['right'].set_visible(False)
h1 = [Patch(color=bar_pal[m], label=m) for m in MODEL_ORDER]
l1 = list(MODEL_ORDER)

# Right: F1, ARI, AMI with SHERLOCK CIs across runs
sns.barplot(
    data=summ_long_ci, x='Metric', y='score',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, width=0.55, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_e2,
)
ax_e2.set_xlabel('')
ax_e2.set_ylabel('Score', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E)
ax_e2.set_ylim(-0.05, 1.05)
ax_e2.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_e2.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_e2.spines['left'].set_linewidth(0.4)
ax_e2.spines['bottom'].set_linewidth(0.4)
ax_e2.tick_params(axis='both', width=0.4)
ax_e2.spines['top'].set_visible(False)
ax_e2.spines['right'].set_visible(False)
ax_e2.get_legend().remove()
ax_e2.legend(h1, l1, title='Model', frameon=False,
             fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E,
             title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_E,
             loc='upper right', bbox_to_anchor=(1, 1.5))

fig_e.suptitle('GI classification', fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_E)
# SHERLOCK vs GEARS significance on the F1/ARI/AMI bars (Welch t, BH-FDR)
add_gi_significance(ax_e2, summ_long_ci, fs=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E)

fig_e.savefig(OUT_DIR / 'panel_e_gi_classification.svg')
plt.show()
print('Panel E saved.')

In [ ]:
# ── Per-run GI classification on the HELD-OUT TEST SET (val_combos) ──────
# Mirror of the per-run cell above, but every metric is restricted to the
# held-out combos only (val_combos == the test set). SHERLOCK reloads each
# checkpoint and re-scores GI; GEARS reuses its cached raw per-run predictions.
# evaluate_gi_classification already returns the full per-combo `work` frame
# (classified, no fit_mask), so we just subset it to the test combos.
_test_suffix     = '_synmerged' if MERGE_SYNERGY else ''
_test_cls_cache  = OUT_DIR / f'gi_test_per_run_class{_test_suffix}.csv'
_test_summ_cache = OUT_DIR / f'gi_test_per_run_summ{_test_suffix}.csv'
_test_combos     = set(val_combos)   # held-out combos == test set

def _subset_gi_metrics(work, model_name, keep_combos, run):
    """Per-class + summary GI metrics on `keep_combos` only (merged synergy)."""
    sub     = work[work.index.isin(keep_combos)]
    eval_df = sub[sub['gt_gi'].notna() & sub['predicted_gi'].notna()].copy()
    y_true  = _merge_syn(eval_df['gt_gi'])
    y_pred  = _merge_syn(eval_df['predicted_gi'])
    labels  = sorted(set(y_true))
    per_class = []
    for cls in labels:
        tb = (y_true == cls).astype(int)
        pb = (y_pred == cls).astype(int)
        n_pos = int(tb.sum())
        auc = roc_auc_score(tb, pb) if 0 < n_pos < len(tb) else np.nan
        per_class.append({
            'model': model_name, 'class': cls, 'n_positive': n_pos,
            'f1': f1_score(tb, pb, zero_division=0), 'auc_roc': auc, 'run': run,
        })
    summary = {
        'model': model_name, 'run': run, 'n_eval': len(eval_df),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'ari': adjusted_rand_score(y_true, y_pred),
        'ami': adjusted_mutual_info_score(y_true, y_pred),
        'nmi': normalized_mutual_info_score(y_true, y_pred),
    }
    return pd.DataFrame(per_class), summary

if (_cache_fresh(_test_cls_cache, *checkpoints.values())
        and _cache_fresh(_test_summ_cache, *checkpoints.values())):
    print('Loading cached per-run TEST-set GI metrics...')
    _test_run_class_df = pd.read_csv(_test_cls_cache)
    _test_run_summ_df  = pd.read_csv(_test_summ_cache)
else:
    _t_cls_rows, _t_summ_rows = [], []
    # SHERLOCK — reload each checkpoint, score GI, restrict to the test combos
    for _rn in range(1, N_RUNS + 1):
        _ckpt = checkpoints[_rn]
        if not _ckpt.exists():
            print(f'  SHERLOCK run {_rn}: checkpoint not found, skipping.')
            continue
        print(f'  SHERLOCK run {_rn}/{N_RUNS}...', end=' ', flush=True)
        _m = slk.tl.load_results(_ckpt)
        slk.tl.eval_single(_m, data, obsm_key='z', uns_key='results', device=DEVICE)
        _syn_r = _m.get_gi(data, min_cells=5)
        _work, _, _ = evaluate_gi_classification(_syn_r, 'SHERLOCK', gi_series)
        _cls_r, _summ_r = _subset_gi_metrics(_work, 'SHERLOCK', _test_combos, _rn)
        _t_cls_rows.append(_cls_r)
        _t_summ_rows.append(_summ_r)
        print('done')
    # GEARS — reuse cached raw per-run predictions
    for _rn in range(1, N_RUNS + 1):
        _g_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv')
        _work, _, _ = evaluate_gi_classification(_g_raw, 'GEARS', gi_series)
        _cls_r, _summ_r = _subset_gi_metrics(_work, 'GEARS', _test_combos, _rn)
        _t_cls_rows.append(_cls_r)
        _t_summ_rows.append(_summ_r)
    _test_run_class_df = pd.concat(_t_cls_rows, ignore_index=True)
    _test_run_summ_df  = pd.DataFrame(_t_summ_rows)
    _test_run_class_df.to_csv(_test_cls_cache, index=False)
    _test_run_summ_df.to_csv(_test_summ_cache, index=False)
    print('Per-run TEST-set GI metrics cached.')

# CI-ready frames for the test-set panel (same shape as class_df_ci / summ_long_ci)
class_df_ci_test = _test_run_class_df.copy()
_slk_test_long = _test_run_summ_df[_test_run_summ_df['model'] == 'SHERLOCK'][['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_slk_test_long['model'] = 'SHERLOCK'
_gears_test_long = _test_run_summ_df[_test_run_summ_df['model'] == 'GEARS'][['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_gears_test_long['model'] = 'GEARS'
summ_long_ci_test = pd.concat(
    [_slk_test_long[['model', 'metric', 'score']],
     _gears_test_long[['model', 'metric', 'score']]],
    ignore_index=True,
)
summ_long_ci_test['Metric'] = pd.Categorical(
    summ_long_ci_test['metric'].map(_SUMM_NAMES),
    categories=list(_SUMM_NAMES.values()), ordered=True,
)
print(f'Test set: {len(_test_combos)} held-out combos')
print(_test_run_summ_df[['model', 'run', 'n_eval', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

In [ ]:
# ── Panel E (test set): GI classification on held-out combos only ────────
# Same layout as Panel E above, but scored on the held-out test combos.
MODEL_ORDER = ['SHERLOCK', 'GEARS']
class_order_test = (
    class_df_ci_test.groupby('class')['auc_roc']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bar_pal = {'SHERLOCK': CLR_SHERLOCK, 'GEARS': CLR_GEARS}

TEXT_SCALE_E = -1.0  # +/- pt added to all Panel E text uniformly; 0.0 == RC defaults

fig_et, (ax_et1, ax_et2) = plt.subplots(
    1, 2, figsize=(300/PPI, 150/PPI), constrained_layout=True,
    gridspec_kw={'width_ratios': [2.2, 1.8]},
)

# Left: per-class AUC-ROC (test set)
sns.barplot(
    data=class_df_ci_test.dropna(subset=['auc_roc']),
    y='class', x='auc_roc',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, order=class_order_test,
    width=0.62, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_et1, legend=False,
)
ax_et1.axvline(0.5, color='0.4', linewidth=0.4, linestyle='--')
ax_et1.set_xlabel('AUC-ROC (one-vs-rest)', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E - 1)
ax_et1.set_ylabel('')
ax_et1.set_yticklabels([_CLASS_LABEL_MAP.get(c, c) for c in class_order_test])
ax_et1.set_xlim(0, 1)
ax_et1.set_xticks([0, 0.5, 1.0])
ax_et1.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_et1.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_et1.spines['left'].set_linewidth(0.4)
ax_et1.spines['bottom'].set_linewidth(0.4)
ax_et1.tick_params(axis='both', width=0.4)
ax_et1.spines['top'].set_visible(False)
ax_et1.spines['right'].set_visible(False)
h1 = [Patch(color=bar_pal[m], label=m) for m in MODEL_ORDER]
l1 = list(MODEL_ORDER)

# Right: F1, ARI, AMI with CIs across runs (test set)
sns.barplot(
    data=summ_long_ci_test, x='Metric', y='score',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, width=0.55, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_et2,
)
ax_et2.set_xlabel('')
ax_et2.set_ylabel('Score', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E)
ax_et2.set_ylim(-0.05, 1.05)
ax_et2.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_et2.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_et2.spines['left'].set_linewidth(0.4)
ax_et2.spines['bottom'].set_linewidth(0.4)
ax_et2.tick_params(axis='both', width=0.4)
ax_et2.spines['top'].set_visible(False)
ax_et2.spines['right'].set_visible(False)
ax_et2.get_legend().remove()
ax_et2.legend(h1, l1, title='Model', frameon=False,
              fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E,
              title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_E,
              loc='upper right', bbox_to_anchor=(1, 1.5))

fig_et.suptitle('GI classification (held-out test set)',
                fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_E)
# SHERLOCK vs GEARS significance on the F1/ARI/AMI bars (Welch t, BH-FDR)
add_gi_significance(ax_et2, summ_long_ci_test, fs=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E)

fig_et.savefig(OUT_DIR / 'panel_e_gi_classification_test.svg')
plt.show()
print('Panel E (test set) saved.')

## GI classification — `observed` mode (singles **and** combos from decoded q(z))

Repeats the two GI classifications above (all data, then held-out test set) but
with SHERLOCK scoring GI in `mode='observed'`: both the single- and the
combination-effect vectors come from the model's decode of the **observed** q(z)
of each perturbation's own cells, with no counterfactual combo shift. GEARS is
unchanged, so this is a like-for-like SHERLOCK ablation of the counterfactual
combo effect used in Panel E. New CI frames get an `_obs` suffix and never
overwrite the counterfactual ones the panels above plot.

In [ ]:
# ── Per-run GI classification — SHERLOCK 'observed' mode (all data) ───────
# Mirror of the all-data per-run cell, but SHERLOCK scores GI with
# mode='observed': singles AND combinations are the decoded observed q(z)
# (no counterfactual combo shift). GEARS is untouched — we reuse the
# merged-synergy per-run frames built above (_gears_run_class_df /
# _gears_run_summ_df). Separate '_observed' caches keep this off the
# counterfactual results.
_obs_suffix     = ('_synmerged' if MERGE_SYNERGY else '') + '_observed'
_obs_cls_cache  = OUT_DIR / f'gi_per_run_class{_obs_suffix}.csv'
_obs_summ_cache = OUT_DIR / f'gi_per_run_summ{_obs_suffix}.csv'

if _cache_fresh(_obs_cls_cache, *checkpoints.values()) and _cache_fresh(_obs_summ_cache, *checkpoints.values()):
    print('Loading cached per-run GI metrics (observed mode)...')
    _run_class_df_obs = pd.read_csv(_obs_cls_cache)
    _run_summ_df_obs  = pd.read_csv(_obs_summ_cache)
else:
    _cls_rows, _summ_rows = [], []
    for _rn in range(1, N_RUNS + 1):
        _ckpt = checkpoints[_rn]
        if not _ckpt.exists():
            print(f'  Run {_rn}: checkpoint not found, skipping.')
            continue
        print(f'  Run {_rn}/{N_RUNS}...', end=' ', flush=True)
        _m = slk.tl.load_results(_ckpt)
        slk.tl.eval_single(_m, data, obsm_key='z', uns_key='results', device=DEVICE)
        _syn_r = _m.get_gi(data, min_cells=5, mode='observed')
        _syn_r.index = (
            _syn_r['pert_a'].astype(str) + '+' + _syn_r['pert_b'].astype(str)
        ).map(canon_pair)
        _syn_r['gt_gi'] = _syn_r.index.map(gi_series)
        _fit_r = _syn_r.index.to_series().isin(set(gi_labels['combination']))
        _syn_r  = classify_gi(_syn_r, fit_mask=_fit_r, inplace=True)
        _, _cls_r, _summ_r = evaluate_gi_classification(_syn_r, 'SHERLOCK', gi_series)
        _cls_r['run']  = _rn
        _summ_r['run'] = _rn
        _cls_rows.append(_cls_r)
        _summ_rows.append(_summ_r)
        print('done')
    _run_class_df_obs = pd.concat(_cls_rows, ignore_index=True)
    _run_summ_df_obs  = pd.DataFrame(_summ_rows)
    _run_class_df_obs.to_csv(_obs_cls_cache, index=False)
    _run_summ_df_obs.to_csv(_obs_summ_cache, index=False)
    print('Per-run GI metrics (observed mode) cached.')

# ── CI-ready frames: SHERLOCK observed + unchanged GEARS ─────────────────
class_df_ci_obs = pd.concat(
    [_run_class_df_obs, _gears_run_class_df], ignore_index=True,
)
_slk_obs_long = _run_summ_df_obs[['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_slk_obs_long['model'] = 'SHERLOCK'
_gears_obs_long = _gears_run_summ_df[['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_gears_obs_long['model'] = 'GEARS'
summ_long_ci_obs = pd.concat(
    [_slk_obs_long[['model', 'metric', 'score']],
     _gears_obs_long[['model', 'metric', 'score']]],
    ignore_index=True,
)
summ_long_ci_obs['Metric'] = pd.Categorical(
    summ_long_ci_obs['metric'].map(_SUMM_NAMES),
    categories=list(_SUMM_NAMES.values()), ordered=True,
)
print('Observed mode (all data) — SHERLOCK:')
print(_run_summ_df_obs[['run', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

In [ ]:
# ── Per-run GI classification on the TEST SET — SHERLOCK 'observed' mode ──
# Held-out combos only (val_combos), SHERLOCK scored with mode='observed'
# (singles + combos from decoded observed q(z); held-out combo cells are still
# present in `data`, so this is measurable — just not OOD). GEARS reuses its
# cached raw per-run predictions, exactly like the counterfactual test cell.
_obs_test_suffix     = ('_synmerged' if MERGE_SYNERGY else '') + '_observed'
_obs_test_cls_cache  = OUT_DIR / f'gi_test_per_run_class{_obs_test_suffix}.csv'
_obs_test_summ_cache = OUT_DIR / f'gi_test_per_run_summ{_obs_test_suffix}.csv'

if (_cache_fresh(_obs_test_cls_cache, *checkpoints.values())
        and _cache_fresh(_obs_test_summ_cache, *checkpoints.values())):
    print('Loading cached per-run TEST-set GI metrics (observed mode)...')
    _test_run_class_df_obs = pd.read_csv(_obs_test_cls_cache)
    _test_run_summ_df_obs  = pd.read_csv(_obs_test_summ_cache)
else:
    _t_cls_rows, _t_summ_rows = [], []
    # SHERLOCK — reload each checkpoint, score GI (observed), restrict to test combos
    for _rn in range(1, N_RUNS + 1):
        _ckpt = checkpoints[_rn]
        if not _ckpt.exists():
            print(f'  SHERLOCK run {_rn}: checkpoint not found, skipping.')
            continue
        print(f'  SHERLOCK run {_rn}/{N_RUNS}...', end=' ', flush=True)
        _m = slk.tl.load_results(_ckpt)
        slk.tl.eval_single(_m, data, obsm_key='z', uns_key='results', device=DEVICE)
        _syn_r = _m.get_gi(data, min_cells=5, mode='observed')
        _work, _, _ = evaluate_gi_classification(_syn_r, 'SHERLOCK', gi_series)
        _cls_r, _summ_r = _subset_gi_metrics(_work, 'SHERLOCK', _test_combos, _rn)
        _t_cls_rows.append(_cls_r)
        _t_summ_rows.append(_summ_r)
        print('done')
    # GEARS — reuse cached raw per-run predictions (unchanged)
    for _rn in range(1, N_RUNS + 1):
        _g_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv')
        _work, _, _ = evaluate_gi_classification(_g_raw, 'GEARS', gi_series)
        _cls_r, _summ_r = _subset_gi_metrics(_work, 'GEARS', _test_combos, _rn)
        _t_cls_rows.append(_cls_r)
        _t_summ_rows.append(_summ_r)
    _test_run_class_df_obs = pd.concat(_t_cls_rows, ignore_index=True)
    _test_run_summ_df_obs  = pd.DataFrame(_t_summ_rows)
    _test_run_class_df_obs.to_csv(_obs_test_cls_cache, index=False)
    _test_run_summ_df_obs.to_csv(_obs_test_summ_cache, index=False)
    print('Per-run TEST-set GI metrics (observed mode) cached.')

# CI-ready frames (observed mode, test set)
class_df_ci_test_obs = _test_run_class_df_obs.copy()
_slk_test_obs_long = _test_run_summ_df_obs[_test_run_summ_df_obs['model'] == 'SHERLOCK'][['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_slk_test_obs_long['model'] = 'SHERLOCK'
_gears_test_obs_long = _test_run_summ_df_obs[_test_run_summ_df_obs['model'] == 'GEARS'][['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_gears_test_obs_long['model'] = 'GEARS'
summ_long_ci_test_obs = pd.concat(
    [_slk_test_obs_long[['model', 'metric', 'score']],
     _gears_test_obs_long[['model', 'metric', 'score']]],
    ignore_index=True,
)
summ_long_ci_test_obs['Metric'] = pd.Categorical(
    summ_long_ci_test_obs['metric'].map(_SUMM_NAMES),
    categories=list(_SUMM_NAMES.values()), ordered=True,
)
print(f'Observed mode — test set: {len(_test_combos)} held-out combos')
print(_test_run_summ_df_obs[['model', 'run', 'n_eval', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

In [ ]:
# ── Panel E (observed mode): GI classification — all data ────────────────
# Same layout as Panel E, but SHERLOCK's GI comes from mode='observed'
# (singles + combos decoded from observed q(z)); GEARS is unchanged.
MODEL_ORDER = ['SHERLOCK', 'GEARS']
class_order_obs = (
    class_df_ci_obs.groupby('class')['auc_roc']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bar_pal = {'SHERLOCK': CLR_SHERLOCK, 'GEARS': CLR_GEARS}

TEXT_SCALE_E = -1.0  # +/- pt added to all Panel E text uniformly; 0.0 == RC defaults

fig_eo, (ax_eo1, ax_eo2) = plt.subplots(
    1, 2, figsize=(300/PPI, 150/PPI), constrained_layout=True,
    gridspec_kw={'width_ratios': [2.2, 1.8]},
)

# Left: per-class AUC-ROC
sns.barplot(
    data=class_df_ci_obs.dropna(subset=['auc_roc']),
    y='class', x='auc_roc',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, order=class_order_obs,
    width=0.62, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_eo1, legend=False,
)
ax_eo1.axvline(0.5, color='0.4', linewidth=0.4, linestyle='--')
ax_eo1.set_xlabel('AUC-ROC (one-vs-rest)', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E - 1)
ax_eo1.set_ylabel('')
ax_eo1.set_yticklabels([_CLASS_LABEL_MAP.get(c, c) for c in class_order_obs])
ax_eo1.set_xlim(0, 1)
ax_eo1.set_xticks([0, 0.5, 1.0])
ax_eo1.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_eo1.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_eo1.spines['left'].set_linewidth(0.4)
ax_eo1.spines['bottom'].set_linewidth(0.4)
ax_eo1.tick_params(axis='both', width=0.4)
ax_eo1.spines['top'].set_visible(False)
ax_eo1.spines['right'].set_visible(False)
h1 = [Patch(color=bar_pal[m], label=m) for m in MODEL_ORDER]
l1 = list(MODEL_ORDER)

# Right: F1, ARI, AMI with CIs across runs
sns.barplot(
    data=summ_long_ci_obs, x='Metric', y='score',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, width=0.55, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_eo2,
)
ax_eo2.set_xlabel('')
ax_eo2.set_ylabel('Score', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E)
ax_eo2.set_ylim(-0.05, 1.05)
ax_eo2.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_eo2.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_eo2.spines['left'].set_linewidth(0.4)
ax_eo2.spines['bottom'].set_linewidth(0.4)
ax_eo2.tick_params(axis='both', width=0.4)
ax_eo2.spines['top'].set_visible(False)
ax_eo2.spines['right'].set_visible(False)
ax_eo2.get_legend().remove()
ax_eo2.legend(h1, l1, title='Model', frameon=False,
              fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E,
              title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_E,
              loc='upper right', bbox_to_anchor=(1, 1.5))

fig_eo.suptitle('GI classification (observed mode)',
                fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_E)
# SHERLOCK vs GEARS significance on the F1/ARI/AMI bars (Welch t, BH-FDR)
add_gi_significance(ax_eo2, summ_long_ci_obs, fs=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E)

fig_eo.savefig(OUT_DIR / 'panel_e_gi_classification_observed.svg')
plt.show()
print('Panel E (observed mode) saved.')

In [ ]:
# ── Panel E (observed mode, test set): GI classification on held-out combos
# Same layout as the observed-mode panel above, restricted to the held-out
# test combos.
MODEL_ORDER = ['SHERLOCK', 'GEARS']
class_order_test_obs = (
    class_df_ci_test_obs.groupby('class')['auc_roc']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bar_pal = {'SHERLOCK': CLR_SHERLOCK, 'GEARS': CLR_GEARS}

TEXT_SCALE_E = -1.0  # +/- pt added to all Panel E text uniformly; 0.0 == RC defaults

fig_eto, (ax_eto1, ax_eto2) = plt.subplots(
    1, 2, figsize=(300/PPI, 150/PPI), constrained_layout=True,
    gridspec_kw={'width_ratios': [2.2, 1.8]},
)

# Left: per-class AUC-ROC (test set)
sns.barplot(
    data=class_df_ci_test_obs.dropna(subset=['auc_roc']),
    y='class', x='auc_roc',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, order=class_order_test_obs,
    width=0.62, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_eto1, legend=False,
)
ax_eto1.axvline(0.5, color='0.4', linewidth=0.4, linestyle='--')
ax_eto1.set_xlabel('AUC-ROC (one-vs-rest)', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E - 1)
ax_eto1.set_ylabel('')
ax_eto1.set_yticklabels([_CLASS_LABEL_MAP.get(c, c) for c in class_order_test_obs])
ax_eto1.set_xlim(0, 1)
ax_eto1.set_xticks([0, 0.5, 1.0])
ax_eto1.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_eto1.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_eto1.spines['left'].set_linewidth(0.4)
ax_eto1.spines['bottom'].set_linewidth(0.4)
ax_eto1.tick_params(axis='both', width=0.4)
ax_eto1.spines['top'].set_visible(False)
ax_eto1.spines['right'].set_visible(False)
h1 = [Patch(color=bar_pal[m], label=m) for m in MODEL_ORDER]
l1 = list(MODEL_ORDER)

# Right: F1, ARI, AMI with CIs across runs (test set)
sns.barplot(
    data=summ_long_ci_test_obs, x='Metric', y='score',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, width=0.55, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_eto2,
)
ax_eto2.set_xlabel('')
ax_eto2.set_ylabel('Score', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E)
ax_eto2.set_ylim(-0.05, 1.05)
ax_eto2.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_eto2.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_eto2.spines['left'].set_linewidth(0.4)
ax_eto2.spines['bottom'].set_linewidth(0.4)
ax_eto2.tick_params(axis='both', width=0.4)
ax_eto2.spines['top'].set_visible(False)
ax_eto2.spines['right'].set_visible(False)
ax_eto2.get_legend().remove()
ax_eto2.legend(h1, l1, title='Model', frameon=False,
               fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E,
               title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_E,
               loc='upper right', bbox_to_anchor=(1, 1.5))

fig_eto.suptitle('GI classification (observed mode, held-out test set)',
                 fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_E)
# SHERLOCK vs GEARS significance on the F1/ARI/AMI bars (Welch t, BH-FDR)
add_gi_significance(ax_eto2, summ_long_ci_test_obs, fs=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E)

fig_eto.savefig(OUT_DIR / 'panel_e_gi_classification_test_observed.svg')
plt.show()
print('Panel E (observed mode, test set) saved.')

### Fig. 5a: genetic-interaction classes in reconstructed gene space

In [ ]:
# ── Panel F: GI Categories in Gene Space ─────────────────────────────────
# One correctly-classified example per GT category (prefer held-out).
# Gene pool: top 100 by variance across all perturbations in treat_effect.
# Genes are hierarchically clustered within each example; no dendrogram shown.
# Also captures _gi_examples for use as panel D arrow annotations.

from scipy.cluster.hierarchy import linkage, leaves_list

def _ch(s):
    parts = str(s).split('+', 1)
    return '+'.join(sorted(p.strip() for p in parts)) if len(parts) == 2 else str(s)

_te_f = data.uns['treat_effect'].to_df()
_te_f.index = _te_f.index.map(_ch)

def _eff_f(pert):
    k = _ch(pert)
    return _te_f.loc[k].values if k in _te_f.index else None

# Top-100 highest-variance genes across all perturbations
_hv100 = np.argsort(_te_f.var(axis=0).values)[-100:]
_gene_names_f = _te_f.columns.to_numpy()[_hv100]

_cats_f = _cats_gi   # defined alongside _gi_examples in the synergy UMAP cell

_nf = len(_gi_examples)

TEXT_SCALE_F = -1  # +/- pt added to all Panel F text uniformly; 0.0 == RC defaults
GENE_FONTSIZE_F = 2
TEXT_SCALE_GENES_F = 0.0  # +/- pt on the x-axis gene labels (scale factor)
GENE_WIDTH_F = 0.14  # figure width in inches allocated per displayed gene
ROW_HEIGHT_PX_F = 128  # figure height (px) per subplot row

# Manual layout (not constrained_layout) so the figure saves at exactly the
# specified figsize and the title / row spacing are directly controllable.
SUBPLOT_TOP_F    = 0.93  # smaller gap between the title and the top row
SUBPLOT_BOTTOM_F = 0.05
SUBPLOT_HSPACE_F = 1.6    # vertical spacing between subfigures (axis-height units)
fig_f, _axes_f_col = plt.subplots(
    _nf,
    1,
    figsize=(600/PPI, 425/PPI),
)
fig_f.subplots_adjust(
    top=SUBPLOT_TOP_F, bottom=SUBPLOT_BOTTOM_F,
    left=0.20, right=0.88, hspace=SUBPLOT_HSPACE_F,
)

_axes_f_flat = np.atleast_1d(_axes_f_col).flatten()

for _i, _cat in enumerate(_cats_f):
    _ax = _axes_f_flat[_i]

    if _cat not in _gi_examples:
        _ax.set_visible(False)
        continue

    _row = _gi_examples[_cat]
    _na, _nb = str(_row['pert_a']), str(_row['pert_b'])
    _combo = _ch(f'{_na}+{_nb}')

    _A = _eff_f(_na)
    _B = _eff_f(_nb)
    _AB = _eff_f(_combo)

    if _A is None or _B is None or _AB is None:
        _ax.text(
            0.5,
            0.5,
            f'missing data\n{_combo}',
            ha='center',
            va='center',
            transform=_ax.transAxes,
            fontsize=7 + TEXT_SCALE_F,
        )
        continue

    _mat = np.stack([
        _A[_hv100],
        _B[_hv100],
        _AB[_hv100],
    ])

    # Cluster columns using each gene's profile across A, B, and A+B.
    # Only the resulting order is used; no dendrogram is drawn.
    _linkage = linkage(
        _mat.T,
        method='average',
        metric='euclidean',
        optimal_ordering=True,
    )
    _gene_order = leaves_list(_linkage)

    _mat = _mat[:, _gene_order]
    _ordered_gene_names = _gene_names_f[_gene_order]

    _vabs = float(np.abs(_mat).max()) or 1.0

    _im = _ax.imshow(
        _mat,
        aspect='auto',
        cmap='RdBu_r',
        vmin=-_vabs,
        vmax=_vabs,
        interpolation='nearest',
    )

    _cbar = plt.colorbar(
        _im,
        ax=_ax,
        shrink=0.9,
        pad=0.01,
    )
    _cbar.ax.tick_params(
        labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_F
    )
    _cbar.set_ticks([-_vabs, 0, _vabs])
    _cbar.set_ticklabels([
        f'{-_vabs:.1f}',
        '0',
        f'{_vabs:.1f}',
    ])

    _ax.set_yticks([0, 1, 2])
    _ax.set_yticklabels(
        [_na, _nb, f'{_na[:5]}+{_nb[:5]}'],
        fontsize=7 + TEXT_SCALE_F,
        fontweight='normal',
        color='0.25',
    )
    _ax.tick_params(axis='y', pad=6)

    # Display every clustered gene name beneath the heatmap
    _ax.set_xticks(np.arange(len(_ordered_gene_names)))
    _ax.set_xticklabels(
        _ordered_gene_names,
        rotation=90,
        ha='center',
        va='top',
        fontsize=GENE_FONTSIZE_F + TEXT_SCALE_GENES_F,
    )
    _ax.tick_params(
        axis='x',
        which='both',
        length=0,
        pad=2,
    )

    _ax.set_title(
        f'{_cat}  |  {_combo}',
        pad=3,
        fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_F - 1,
    )

fig_f.text(
    0.985,
    0.5,
    'Δ Gene Expression',
    rotation=270,
    va='center',
    ha='right',
    fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_F,
)

fig_f.suptitle(
    'GI Categories in Reconstructed Gene Space',
    fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_F,
    y=0.99,
)

fig_f.savefig(
    OUT_DIR / 'panel_f_gi_heatmap.svg',
)

plt.show()

# Build arrow dict for panel D
_gi_arrows = {}

for _cat, _row in _gi_examples.items():
    _na2 = str(_row['pert_a'])
    _nb2 = str(_row['pert_b'])
    _cidx = _ch(f'{_na2}+{_nb2}')

    if _cidx in syn_umap.index:
        _lbl = f'{_na2[:5]}/{_nb2[:5]}'
        _gi_arrows[_lbl] = _cidx

print(f'Panel F saved. Arrow pairs: {list(_gi_arrows.keys())}')

## Consensus modules and the erythroid axis

From [`scripts/norman/4_interaction_biology.ipynb`](../scripts/norman/4_interaction_biology.ipynb).

Needs `results/module_interaction/consensus_modules.csv`, written by [`scripts/norman/3_module_interaction.ipynb`](../scripts/norman/3_module_interaction.ipynb).

# Norman 2019 — interaction rules and the erythroid axis

K562 is a chronic myeloid leukaemia (CML) blast-crisis line that can be pushed toward erythroid,
megakaryocytic or granulocytic states; Norman et al. describe erythroid, megakaryocyte (CD41⁺) and
granulocyte clusters in this screen. Using the 5 trained SHERLOCK runs from `norman_figure.ipynb`, this
notebook:

1. rebuilds the seven model-parameter descriptors and the rule-based interaction classifier;
2. finds which latent (W) axes carry the erythroid and granulocytic programmes;
3. defines the fate axis from canonical lineage markers and places the modules on it;
4. tests how the positions of a pair's two modules relate to its interaction type.

Only the **measured, labelled panel** is analysed — a curated selection of combinations, so nothing is
extrapolated to unmeasured pairs. Nothing is written to disk. Modules are the consensus modules from
`module_interaction.ipynb`; ground truth is Norman's `genetic_interactions.csv` with the two synergy
classes merged.

In [ ]:
import sys, warnings
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd, torch, scanpy as sc
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from scipy.stats import mannwhitneyu
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
warnings.filterwarnings('ignore')
import sherlock as slk

mpl.rcParams.update({'font.size': 8, 'axes.titlesize': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 150, 'svg.fonttype': 'none'})
N_RUNS, GATE_THR, N_BG = 5, 0.5, 256
FIG = Path(f'{paths.RESULTS_DIR}/norman_figure')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PALETTE = {'approximately additive': '#4DAF4A', 'epistasis': '#FF7F00', 'neomorphic': '#984EA3',
           'potentiation': '#F781BF', 'redundant': '#A65628', 'suppression': '#E41A1C', 'synergy': '#377EB8'}
CLASSES = list(PALETTE)
short = lambda c: c.replace('approximately additive', 'additive')
NAMES = {'rho': 'parent similarity', 'private_frac': 'private latent factors',
         'c_sum': 'parent scaling', 'syn_rel': 'interaction term size',
         'joint_over_add': 'combination vs sum of parents', 'dom_gap': 'parent dominance',
         'syn_offgate_dec': 'interaction on unused factors'}

# data, with the gene filter the models were trained on
data = sc.read_h5ad(f'{paths.DATA_DIR}/norman/Norman_2019.h5ad')
data.X = data.layers['counts'].copy()
slk.pp.slk_prepare_data(data, pert_key='perturbation_name', ntc_label='control', treatment_key=None, inplace=True)
labels = data.obs[slk.configs.get_config('pert_key')].astype(str).values
NTC = slk.configs.get_config('ntc_label')
genes = sorted({g for l in set(labels) if l != NTC for g in l.split('+')})
g2i = {g: i for i, g in enumerate(genes)}
P = len(genes)
_lib = np.asarray(data.X.sum(1)).ravel()
def _pb(mask):                                      # observed pseudobulk log2 CPM of the current gene set
    return np.log2(np.asarray(data.X[mask].multiply(1e4 / _lib[mask][:, None]).mean(0)).ravel() + 1)

# Observed fate scores from CANONICAL markers, on all genes (before the model's gene filter).
# Per marker: log2 fold change vs NTC for each single perturbation, z-scored across perturbations;
# a fate score is the mean over that fate's markers. CRISPRa targets are never used as markers.
MARKERS = {
    'erythroid': ['HBG1', 'HBG2', 'HBZ', 'HBA1', 'HBA2', 'HBE1', 'HBD', 'GYPA', 'GYPB', 'ALAS2', 'SLC25A37',
                  'TFRC', 'KLF1', 'GATA1', 'EPOR', 'AHSP', 'SLC4A1'],
    'megakaryocytic': ['ITGA2B', 'ITGB3', 'GP9', 'GP1BA', 'GP1BB', 'GP6', 'PF4', 'PPBP', 'VWF', 'TUBB1', 'SELP',
                       'CD9', 'MYL9', 'TREML1', 'F13A1', 'PLEK', 'MPL', 'LTBP1', 'CD36', 'PECAM1'],
    'granulocytic': ['MPO', 'ELANE', 'AZU1', 'PRTN3', 'CTSG', 'LYZ', 'S100A8', 'S100A9', 'CSF3R', 'ITGAM', 'CD33',
                     'MNDA', 'FCGR1A', 'LST1', 'FCER1G', 'CST3', 'CEBPE', 'LGALS3', 'ANXA1', 'SRGN'],
}
FATES = list(MARKERS)
MARKERS = {f: [g for g in gs if g in set(data.var_names) and g not in set(genes)] for f, gs in MARKERS.items()}
_mg = sorted({g for gs in MARKERS.values() for g in gs})
_Xm = data[:, _mg].X.tocsr()
_cpm = lambda mask: np.asarray(_Xm[mask].multiply(1e4 / _lib[mask][:, None]).mean(0)).ravel()
_ntc_m = _cpm(labels == NTC)
_lfc = pd.DataFrame(np.stack([np.log2(_cpm(labels == g) + .05) - np.log2(_ntc_m + .05) for g in genes]),
                    index=genes, columns=_mg)
_z = (_lfc - _lfc.mean()) / (_lfc.std() + 1e-9)
fate = pd.DataFrame({f: _z[gs].mean(1) for f, gs in MARKERS.items()})
_det = np.asarray((_Xm[labels == NTC] > 0).mean(0)).ravel()
print('canonical markers used:', {f: len(gs) for f, gs in MARKERS.items()},
      '| detected in control cells (median):', {f: round(float(np.median(_det[[_mg.index(g) for g in gs]])), 3)
                                                 for f, gs in MARKERS.items()})
data = data[:, (np.asarray(data.X.mean(0)).ravel() > 0.5) | data.var_names.isin(set(genes))].copy()
var = np.array(data.var_names)
readout = ~np.isin(var, genes)                      # exclude CRISPRa targets from any readout
bg_idx = np.random.default_rng(0).choice(np.where(labels == NTC)[0], N_BG, replace=False)
x_bg = torch.tensor(data.X[bg_idx].toarray(), dtype=torch.float32, device=DEVICE)

# learned rho (5 runs), consensus modules, ground truth
_m = np.load(FIG / 'run_matrices.npz')
R = np.mean([(_m[f'rho_corr_{k}'] + _m[f'rho_corr_{k}'].T) / 2 for k in range(1, N_RUNS + 1)], 0)
module = pd.read_csv(f'{paths.RESULTS_DIR}/module_interaction/consensus_modules.csv', index_col=0)['module']
MODULES = sorted(module.unique(), key=lambda m: int(m[1:]))
gi = pd.read_csv(f'{paths.DATA_DIR}/norman/genetic_interactions.csv')
gi['combination'] = gi['combination'].map(lambda s: '+'.join(sorted(s.split('+'))))
gi['gi'] = gi['genetic_interaction'].str.strip().replace({'synergy (dissimilar phenotype)': 'synergy',
                                                          'synergy (similar phenotype)': 'synergy'})
gi = gi[gi['combination'].map(lambda c: all(g in g2i for g in c.split('+')))].reset_index(drop=True)
print(f'{P} perturbations, {readout.sum()} readout genes, {len(gi)} labelled pairs, {len(MODULES)} modules')

## 1. Seven descriptors from the model's parameters

Computed for the labelled pairs only. The combined latent state is
**z_ab = z₀ + c_a·S_a + c_b·S_b + syn(ρ_a, ρ_b)** with S = A⊙W.

| Descriptor | From | Meaning |
|---|---|---|
| `rho` | covariance Σ_P | do the two perturbations act in the same direction? |
| `private_frac` | gates W, A⊙W | share of each shift on factors the partner does not gate |
| `c_sum` | parent-scale layer | c_a + c_b: damped (< 2) or amplified |
| `syn_rel` | synergy MLP | ‖syn‖ / ‖S_a + S_b‖ — how big the synergy term is |
| `joint_over_add` | decoder | ‖decoded combination‖ / ‖D_a + D_b‖ |
| `dom_gap` | decoder | \|cos(combination, D_a) − cos(combination, D_b)\| — dominance |
| `syn_offgate_dec` | syn × W × decoder | decoded effect of syn on factors **neither** parent gates — where it acts |

In [ ]:
lab = gi[['combination', 'gi', 'genetic_interaction']].copy()
lab['a'] = lab['combination'].str.split('+').str[0]
lab['b'] = lab['combination'].str.split('+').str[1]
IA, IB = lab['a'].map(g2i).values, lab['b'].map(g2i).values
DESC = ['private_frac', 'c_sum', 'syn_rel', 'joint_over_add', 'dom_gap', 'syn_offgate_dec']
runs = []
for run in range(1, N_RUNS + 1):
    m = slk.tl.load_results(FIG / 'models' / f'run_{run}.pth').to(DEVICE)
    with torch.no_grad():
        sh = m._get_all_pert_shifts(DEVICE)[1:]
        S, W, q = sh[:, 0], sh[:, 1], sh[:, 2]
        u = m._abduct_ntc(x_bg)
        ln = lambda z: torch.log1p(m._decode_to_expr(z, 1.) * 1e4)
        def dec(shift):                                            # (b, d) -> (b, G), mean over backgrounds
            z = (u[None] + shift[:, None]).reshape(-1, u.shape[-1])
            return ln(z).reshape(shift.shape[0], N_BG, -1).mean(1)
        base = ln(u).mean(0)
        D = dec(S) - base                                          # single-perturbation effects
        act = W > GATE_THR
        delta = float(S.abs()[act].median())                       # decoded signature of each latent factor
        E = torch.eye(m.latent_dim, device=DEVICE) * delta
        V = ((dec(E) - dec(-E)) / 2).cpu().numpy()
        a = torch.as_tensor(IA, device=DEVICE); b = torch.as_tensor(IB, device=DEVICE)
        ca, cb = m._parent_scales_from_rho(q[a], q[b])
        syn = m._synergy_shift_from_rho(q[a], q[b])
        lin = ca * S[a] + cb * S[b]
        g_lin = dec(lin)
        d_ab = dec(lin + syn) - base
        off = ~(act[a] | act[b])
        add = D[a] + D[b]; addn = add.norm(dim=1) + 1e-8
        cos = lambda x, y: (x * y).sum(1) / (x.norm(dim=1) * y.norm(dim=1) + 1e-8)
        pa = ((~act[b]) * S[a] ** 2).sum(1) / ((S[a] ** 2).sum(1) + 1e-8)
        pb = ((~act[a]) * S[b] ** 2).sum(1) / ((S[b] ** 2).sum(1) + 1e-8)
        desc = {'private_frac': (pa + pb) / 2, 'c_sum': (ca + cb).squeeze(-1),
                'syn_rel': syn.norm(dim=1) / ((S[a] + S[b]).norm(dim=1) + 1e-8),
                'joint_over_add': d_ab.norm(dim=1) / addn,
                'dom_gap': (cos(d_ab, D[a]) - cos(d_ab, D[b])).abs(),
                'syn_offgate_dec': (dec(lin + syn * off) - g_lin).norm(dim=1) / addn}
        runs.append({'desc': pd.DataFrame({k: v.cpu().numpy() for k, v in desc.items()}),
                     'D': D.cpu().numpy(), 'S': S.cpu().numpy(), 'W': W.cpu().numpy(), 'V': V})
    del m; torch.cuda.empty_cache()

for k in DESC:
    lab[k] = np.mean([r['desc'][k].values for r in runs], 0)
lab['rho'] = R[IA, IB]
lab['module_a'] = lab['a'].map(module); lab['module_b'] = lab['b'].map(module)
DESC7 = ['rho'] + DESC
print(f'descriptors computed for the {len(lab)} labelled pairs')

## 2. Rule-based classification

A decision list, applied top to bottom; unclaimed pairs are called additive. The cut points are the
fixed values found by rule mining in `module_interaction.ipynb`.

In [ ]:
RULES = [  # (class, [(descriptor, op, threshold)])
    ('approximately additive', [('c_sum', '<=', 1.8769652366638183), ('syn_rel', '<=', 0.1731259450316429)]),
    ('redundant',    [('rho', '>', 0.36017714440822585), ('joint_over_add', '<=', 0.9664641320705414)]),
    ('synergy',      [('rho', '>', 0.36017714440822585), ('joint_over_add', '>', 1.1006439685821534)]),
    ('potentiation', [('private_frac', '>', 0.24869043231010438), ('c_sum', '>', 1.502985668182373)]),
    ('epistasis',    [('joint_over_add', '<=', 1.0306628942489624), ('dom_gap', '>', 0.6432075917720794)]),
    ('suppression',  [('rho', '<=', 0.13503594752401113), ('joint_over_add', '<=', 1.1355479955673218)]),
    ('neomorphic',   [('syn_offgate_dec', '>', 0.06492092683911324), ('rho', '<=', 0.3258450113236904)]),
]

def classify(df):
    pred = np.array(['approximately additive'] * len(df), dtype=object)
    free = np.ones(len(df), bool)
    for cls, conds in RULES:
        hit = free.copy()
        for f, op, thr in conds:
            hit &= (df[f] > thr).values if op == '>' else (df[f] <= thr).values
        pred[hit] = cls; free &= ~hit
    return pred

lab['rule_class'] = classify(lab)
acc = np.mean(lab['rule_class'] == lab['gi']); bal = balanced_accuracy_score(lab['gi'], lab['rule_class'])
print(f'Rule-based classifier on {len(lab)} labelled pairs: accuracy {acc:.3f}, balanced {bal:.3f}')
for cls, conds in RULES:
    print(f"  {short(cls):12s}: " + '  AND  '.join(f'{f} {op} {thr:.3g}' for f, op, thr in conds))

auc = pd.DataFrame({f: {c: mannwhitneyu(lab.loc[lab.gi == c, f], lab.loc[lab.gi != c, f]).statistic /
                        ((lab.gi == c).sum() * (lab.gi != c).sum()) for c in CLASSES} for f in DESC7})
fig, ax = plt.subplots(1, 2, figsize=(13, 3.9), gridspec_kw={'width_ratios': [1, 1.3]})
sns.heatmap(confusion_matrix(lab['gi'], lab['rule_class'], labels=CLASSES), annot=True, fmt='d', cmap='Oranges',
            cbar=False, linewidths=1, linecolor='white', ax=ax[0],
            xticklabels=[short(c) for c in CLASSES], yticklabels=[short(c) for c in CLASSES])
ax[0].set(xlabel='class assigned by the rules', ylabel='Norman label',
          title=f'Rule classifier, 86 labelled pairs: accuracy {acc:.2f}, balanced accuracy {bal:.2f}')
sns.heatmap(auc.rename(columns=NAMES), annot=True, fmt='.2f', cmap='PuOr_r', center=.5, vmin=0, vmax=1, linewidths=.5,
            linecolor='white', ax=ax[1], yticklabels=[short(c) for c in CLASSES], cbar_kws={'label': 'AUROC'})
ax[1].set_title('Separation of each class by each model parameter\n(AUROC; 0.5 = no separation, > 0.5 = higher in that class)')
for a_ in ax:
    a_.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

## 3. Which W axes carry the erythroid and granulocytic programmes?

Each latent factor's decoded signature (the expression change from moving along that axis) is matched
across runs (Hungarian assignment on |cos|, sign-corrected) and scored on two programmes, using only
canonical markers the model decodes:

- **erythroid** — globins, glycophorins, haem synthesis, GATA1, NFE2, TFRC (20 genes);
- **granulocytic** — CD33, LST1, FCER1G, CST3 (the only decodable canonical granulocyte markers).

Canonical megakaryocytic markers (ITGA2B/CD41, ITGB3, GP9, PF4, …) are all below the model's gene
filter, so megakaryocytic priming is assessed from the observed data in section 4, not from model axes.

In [ ]:
ERY = ['HBG1', 'HBG2', 'HBZ', 'HBA1', 'HBE1', 'HBD', 'HBQ1', 'GYPA', 'GYPB', 'GYPC', 'ALAS2', 'HMBS', 'UROD',
       'BLVRB', 'SLC25A37', 'GATA1', 'NFE2', 'TFRC', 'ANK1', 'PRDX2']
GRAN = [g for g in MARKERS['granulocytic'] if g in set(var[readout])]
PROGS = {'erythroid': np.isin(var, ERY) & readout, 'granulocytic': np.isin(var, GRAN) & readout}
P2 = list(PROGS)
print('decodable programme genes:', {k: int(v.sum()) for k, v in PROGS.items()}, '| granulocytic:', GRAN)
def prog_score(M):                                  # gene-space rows -> programme scores (per-row z units)
    sd = M[:, readout].std(1) + 1e-9
    return pd.DataFrame({k: M[:, msk].mean(1) / sd for k, msk in PROGS.items()})

nrm = lambda X: X[:, readout] / (np.linalg.norm(X[:, readout], axis=1, keepdims=True) + 1e-9)
L = runs[0]['V'].shape[0]
perm, sign, match = [np.arange(L)], [np.ones(L)], []
for r in runs[1:]:
    C = nrm(runs[0]['V']) @ nrm(r['V']).T
    ri, ci = linear_sum_assignment(-np.abs(C))
    perm.append(ci); sign.append(np.sign(C[ri, ci])); match.append(np.abs(C[ri, ci]))
V_al = np.stack([r['V'][p] * sg[:, None] for r, p, sg in zip(runs, perm, sign)])
S_al = np.stack([r['S'][:, p] * sg for r, p, sg in zip(runs, perm, sign)])
W_al = np.stack([r['W'][:, p] for r, p in zip(runs, perm)])

per_run = [prog_score(V) for V in V_al]
fac = sum(per_run) / N_RUNS
fac.index = [f'F{k}' for k in range(L)]
for p_ in P2:
    fac[f'sd_{p_}'] = np.std([s_[p_] for s_ in per_run], 0)
fac['cross-run similarity'] = np.mean(match, 0)
fac['similarity sd'] = np.std(match, 0)
_nuse = (W_al > GATE_THR).sum(1)                    # perturbations using each factor, per run
fac['perts using'] = _nuse.mean(0)
fac['perts using sd'] = _nuse.std(0)
used = fac['perts using'] >= 1
dom = fac[P2].abs().idxmax(1)
val = fac[P2].to_numpy()[np.arange(L), [P2.index(d) for d in dom]]
fac['call'] = np.where(~used, 'unused', np.where(np.abs(val) < 0.5, 'none', dom + np.where(val > 0, ' +', ' −')))
load = pd.DataFrame(S_al.mean(0), index=genes, columns=fac.index)
fac['drives +'] = [', '.join(load[f].nlargest(4).index) for f in fac.index]
fac['drives −'] = [', '.join(load[f].nsmallest(4).index) for f in fac.index]
print(f"{(~used).sum()} of {L} latent factors are used by no perturbation and are excluded: {', '.join(fac.index[~used])}")
fac[used].sort_values('erythroid', ascending=False)[P2 + ['cross-run similarity', 'similarity sd',
    'perts using', 'perts using sd', 'call', 'drives +', 'drives −']].round(2)

### Can the latent axes be matched across runs?

Latent dimensions are arbitrary within a run, so the axis names above rely on matching. **(a)** For
runs 2–5, the |cosine| between every run-1 axis signature and every signature of that run, with columns
reordered by the matching: a bright, clean diagonal means each axis has one clear counterpart.
**(b)** For every axis, the |cosine| of its match in each run (dots), the best alternative in the same run
(grey ×), and the range of |cosine| between unmatched axes (shaded; 95th percentile marked). Unused axes
are shown in grey. The permutation test compares the total matched |cosine| with 5,000 random pairings.

In [ ]:
Cs = []
for r, p_ in zip(runs[1:], perm[1:]):
    Cs.append(np.abs(nrm(runs[0]['V']) @ nrm(r['V']).T)[:, p_])     # matched partners on the diagonal
eye = np.eye(L, dtype=bool)
diag = np.stack([np.diag(Cm) for Cm in Cs])                          # (runs 2-5, axes)
runner = np.stack([np.where(eye, -1, Cm).max(1) for Cm in Cs])       # best alternative for each run-1 axis
offd = np.concatenate([Cm[~eye] for Cm in Cs])
null95 = np.quantile(offd, .95)
rng_m = np.random.default_rng(0)
perm_p = []
for Cm in Cs:
    tot = np.trace(Cm)
    null_tot = np.array([Cm[np.arange(L), rng_m.permutation(L)].sum() for _ in range(5000)])
    perm_p.append((np.sum(null_tot >= tot) + 1) / 5001)
match_tab = pd.DataFrame({'mean matched |cos|': diag.mean(0), 'lowest matched |cos|': diag.min(0),
                          'mean best alternative': runner.mean(0), 'mean margin': (diag - runner).mean(0),
                          'runs above null 95th pct': (diag > null95).sum(0), 'used': used.values},
                         index=fac.index)
print(f'Unmatched pairs: median |cos| {np.median(offd):.2f}, 95th percentile {null95:.2f}')
print('Total matched |cos| vs random pairings, per run:',
      ', '.join(f'run {k + 2}: p = {pv:.4f}' for k, pv in enumerate(perm_p)))

fig = plt.figure(figsize=(15, 8.2))
gsm = fig.add_gridspec(2, 5, height_ratios=[1, 1.05], width_ratios=[1, 1, 1, 1, .06], hspace=.45, wspace=.12)
cax = fig.add_subplot(gsm[0, 4])
for k, Cm in enumerate(Cs):
    axm = fig.add_subplot(gsm[0, k])
    sns.heatmap(Cm, cmap='viridis', vmin=0, vmax=1, square=True, cbar=k == len(Cs) - 1, ax=axm,
                cbar_ax=cax if k == len(Cs) - 1 else None,
                xticklabels=False, yticklabels=list(fac.index) if k == 0 else False,
                cbar_kws={'label': '|cos|'} if k == len(Cs) - 1 else None)
    axm.set_title(f'run 1 vs run {k + 2}\n(total matched vs random: p = {perm_p[k]:.4f})', fontsize=8)
    axm.set_xlabel(f'run {k + 2} latent factors (matched order)', fontsize=7)
    axm.tick_params(axis='y', labelsize=6)
fig.axes[1].set_ylabel('run 1 latent factors', fontsize=8)
axb = fig.add_subplot(gsm[1, :4])
xs = np.arange(L)
axb.axhspan(offd.min(), null95, color='0.9', zorder=0)
axb.axhline(null95, color='0.5', ls='--', lw=.8)
axb.text(L - .5, null95 + .01, 'unmatched pairs, 95th percentile', ha='right', va='bottom', fontsize=7, color='0.35')
for j in range(L):
    col = '#2166ac' if used.iloc[j] else '0.65'
    axb.scatter(np.full(diag.shape[0], j) + np.linspace(-.15, .15, diag.shape[0]), diag[:, j], s=26, color=col, zorder=3)
    axb.scatter(np.full(diag.shape[0], j) + np.linspace(-.15, .15, diag.shape[0]), runner[:, j], s=22, marker='x',
                color='0.55', lw=1, zorder=2)
    axb.hlines(diag[:, j].mean(), j - .3, j + .3, color=col, lw=2, zorder=4)
axb.set_xticks(xs)
axb.set_xticklabels([f"{f}{'' if u else ' (unused)'}" for f, u in zip(fac.index, used)], rotation=40, ha='right', fontsize=7)
axb.set_ylim(0, 1.02); axb.set_ylabel('signature similarity with the run-1 factor (|cos|)')
axb.legend(handles=[mpl.lines.Line2D([], [], marker='o', ls='', color='#2166ac', label='matched factor, one dot per run'),
                    mpl.lines.Line2D([], [], marker='x', ls='', color='0.55', label='closest other factor in that run'),
                    mpl.lines.Line2D([], [], marker='o', ls='', color='0.65', label='factor used by no perturbation')],
           frameon=False, fontsize=7, loc='lower left', ncol=3)
axb.set_title('Match quality per latent factor', fontsize=9, loc='left')
axb.text(-0.045, 1.02, 'b', transform=axb.transAxes, fontsize=15, va='bottom', ha='right')
fig.axes[1].text(-0.25, 1.06, 'a', transform=fig.axes[1].transAxes, fontsize=15, va='bottom', ha='right')
plt.show()
match_tab.round(2)

## 4. Defining the fate axis

Each perturbation gets an **observed** fate score per lineage from canonical markers (section 1 setup).
These give the axis a biological definition:

- **erythroid programme up vs down** is the main axis; the model's decoded erythroid score agrees with it;
- **megakaryocytic** priming (canonical markers) is examined separately, because it does not oppose
  the erythroid programme in K562. It is a **weak signal**: its markers are detected in ~1% of control
  cells and are absent from the model's genes, so the code below checks that the score is reproducible
  (split-half reliability) and puts a confidence interval on its relation to the erythroid score;
- **granulocytic** priming is what opposes it.

Modules are assigned to a **pole** by their members' mean observed erythroid score: *erythroid up*
(> +0.2), *erythroid down* (< −0.2) or *neutral*. Within the *down* pole, a module is called
*granulocytic* if its granulocytic score exceeds 0.5; within the *up* pole, *erythroid + megakaryocytic*
if its megakaryocytic score exceeds 0.3.

In [ ]:
from scipy.stats import pearsonr, spearmanr, fisher_exact
print('Observed canonical fate scores: correlation across the 105 single perturbations')
print(fate.corr().round(2).to_string())

# How trustworthy is each score? Split every perturbation's cells (and the controls) into two random halves,
# score each half separately, and correlate the halves across perturbations (10 random splits).
rng_f = np.random.default_rng(0)
def _fate_scores(rows_by_label):
    ntc_ = _cpm(rows_by_label[NTC])
    lf = pd.DataFrame(np.stack([np.log2(_cpm(rows_by_label[g]) + .05) - np.log2(ntc_ + .05) for g in genes]),
                      index=genes, columns=_mg)
    zz = (lf - lf.mean()) / (lf.std() + 1e-9)
    return pd.DataFrame({f: zz[gs].mean(1) for f, gs in MARKERS.items()})
_rows = {g: np.where(labels == g)[0] for g in genes + [NTC]}
half_r = {f: [] for f in FATES}
for _ in range(10):
    split = {g: rng_f.permutation(ix) for g, ix in _rows.items()}
    h1 = _fate_scores({g: ix[: len(ix) // 2] for g, ix in split.items()})
    h2 = _fate_scores({g: ix[len(ix) // 2:] for g, ix in split.items()})
    for f in FATES:
        half_r[f].append(pearsonr(h1[f], h2[f])[0])
rel = pd.DataFrame({'split-half r': {f: np.mean(v) for f, v in half_r.items()}})
rel['full-data reliability'] = 2 * rel['split-half r'] / (1 + rel['split-half r'])     # Spearman-Brown
print('\nScore reliability:'); print(rel.round(2).to_string())

x_, y_ = fate['erythroid'].values, fate['megakaryocytic'].values
r_em = pearsonr(x_, y_)[0]
boot = [pearsonr(x_[i], y_[i])[0] for i in rng_f.integers(0, len(x_), (5000, len(x_)))]
permr = np.array([pearsonr(x_, rng_f.permutation(y_))[0] for _ in range(5000)])
keep = fate.index != 'FEV'
print(f"\nErythroid vs megakaryocytic: r = {r_em:.2f}, 95% CI [{np.quantile(boot, .025):.2f}, "
      f"{np.quantile(boot, .975):.2f}], permutation p = {(np.sum(np.abs(permr) >= abs(r_em)) + 1) / 5001:.4f}; "
      f"without FEV r = {pearsonr(x_[keep], y_[keep])[0]:.2f}")

D_mean = np.mean([r['D'] for r in runs], 0)
pert = prog_score(D_mean); pert.index = genes
print(f"\nModel-decoded vs observed erythroid score: r = {pearsonr(pert['erythroid'], fate['erythroid'])[0]:.2f}"
      f" (Spearman {spearmanr(pert['erythroid'], fate['erythroid'])[0]:.2f})")

NORMAN = {'erythroid': ['CBL', 'CNN1', 'UBASH3A', 'UBASH3B', 'PTPN9', 'PTPN12'],
          'granulocyte': ['CEBPA', 'CEBPB', 'CEBPE', 'SPI1'],
          'G1 arrest': ['CDKN1A', 'CDKN1B', 'CDKN1C']}
print('\nNorman’s named groups (observed fate scores, mean over members):')
print(pd.DataFrame({k: fate.loc[v].mean() for k, v in NORMAN.items()}).T.round(2).to_string())
print('\nStrongest canonical megakaryocytic priming (Norman’s CD41⁺ group has no named members):')
print(fate.sort_values('megakaryocytic', ascending=False).head(6).round(2).to_string())

mods = fate.groupby(module.reindex(genes).values).mean().reindex(MODULES)
mods['decoded erythroid'] = pert.groupby(module.reindex(genes).values)['erythroid'].mean()
mods['n'] = module.value_counts().reindex(MODULES)
mods['pole'] = np.select([mods['erythroid'] > .2, mods['erythroid'] < -.2], ['erythroid up', 'erythroid down'], 'neutral')
mods['identity'] = np.select(
    [(mods['pole'] == 'erythroid down') & (mods['granulocytic'] > .5), mods['pole'] == 'erythroid down',
     (mods['pole'] == 'erythroid up') & (mods['megakaryocytic'] > .3), mods['pole'] == 'erythroid up'],
    ['granulocytic', 'erythroid repressed only', 'erythroid + megakaryocytic', 'erythroid'], 'neutral')
for k, v in NORMAN.items():
    mods[f'Norman {k}'] = [', '.join(sorted(set(module.index[module == m]) & set(v))) for m in MODULES]
s6_groups = {}
for df in pd.read_excel(Path(paths.DATA_DIR) / 'norman' / 'aax4438_tables6.xlsx', sheet_name=None, header=None).values():
    rows = df.values.tolist()
    for i, rr in enumerate(rows[:-1]):
        if str(rr[0]).strip().isdigit():
            s6_groups[str(rr[1]).strip()] = [g.strip() for g in str(rows[i + 1][0]).split(',')]
mods['Norman Table S6 clusters (≥ 2 members)'] = [
    '; '.join(sorted({lbl for lbl, gs in s6_groups.items() if len(set(gs) & set(module.index[module == m])) >= 2}))
    for m in MODULES]
mods['members'] = [', '.join(module.index[module == m]) for m in MODULES]
pd.set_option('display.max_colwidth', 80)
mods.sort_values('erythroid', ascending=False).round(2)

## 5. How a pair's position on the axis relates to its interaction

Two views of the labelled pairs: the **poles of the two modules** (erythroid up / neutral / down,
abbreviated U / 0 / D) and the **observed erythroid scores of the two parents**. Enrichment is tested
with Fisher's exact test against the other labelled pairs, and the eight tests are adjusted together
(Benjamini–Hochberg q). These statistics describe this panel only.

In [ ]:
PS = {'erythroid up': 'U', 'neutral': '0', 'erythroid down': 'D'}
lab['pole_pair'] = ['×'.join(sorted([PS[mods.loc[module[a], 'pole']], PS[mods.loc[module[b], 'pole']]],
                                    key=['U', '0', 'D'].index)) for a, b in zip(lab['a'], lab['b'])]
lab['within_module'] = lab['module_a'] == lab['module_b']
lab['ery_lo'] = [min(fate.loc[a, 'erythroid'], fate.loc[b, 'erythroid']) for a, b in zip(lab['a'], lab['b'])]
lab['ery_hi'] = [max(fate.loc[a, 'erythroid'], fate.loc[b, 'erythroid']) for a, b in zip(lab['a'], lab['b'])]
lab['ery_gap'] = lab['ery_hi'] - lab['ery_lo']
lab['opposite_parents'] = (lab['ery_lo'] < 0) & (lab['ery_hi'] > 0)
PP = [p for p in ['U×U', 'U×0', 'U×D', '0×0', '0×D', 'D×D'] if p in set(lab['pole_pair'])]
ct_pole = pd.crosstab(lab['pole_pair'], lab['gi']).reindex(index=PP, columns=CLASSES, fill_value=0)
print('Interactions by the poles of the two modules (U = erythroid up, D = erythroid down):')
print(ct_pole.assign(n=ct_pole.sum(1)).to_string())

def enrich(cls, mask, label):
    k = int(((lab['gi'] == cls) & mask).sum()); n = int((lab['gi'] == cls).sum())
    t = [[k, n - k], [int(mask.sum()) - k, len(lab) - int(mask.sum()) - (n - k)]]
    orr, p = fisher_exact(t)
    return {'class': short(cls), 'where': label, 'in': f'{k}/{n}', 'base rate': f'{mask.mean():.0%}',
            'odds ratio': orr, 'p': p}
claims = pd.DataFrame([
    enrich('redundant', lab['within_module'], 'same module'),
    enrich('suppression', lab['within_module'], 'same module'),
    enrich('suppression', lab['pole_pair'] == 'U×D', 'opposite poles (U×D)'),
    enrich('suppression', lab['opposite_parents'], 'parents push erythroid opposite ways'),
    enrich('redundant', lab['pole_pair'] == 'U×D', 'opposite poles (U×D)'),
    enrich('approximately additive', lab['pole_pair'] == 'U×D', 'opposite poles (U×D)'),
    enrich('synergy', lab['pole_pair'] == 'U×U', 'both erythroid up (U×U)'),
    enrich('redundant', lab['pole_pair'] == 'D×D', 'both erythroid down (D×D)'),
])
from statsmodels.stats.multitest import multipletests
claims['q (BH)'] = multipletests(claims['p'], method='fdr_bh')[1]
claims = claims.drop(columns='p')
print('\n' + claims.round(4).to_string(index=False))

opp = lab[(lab['pole_pair'] == 'U×D') & lab['gi'].isin(['synergy', 'suppression'])]
cmp_ = opp.groupby('gi')[['ery_gap', 'rho', 'joint_over_add', 'c_sum']].mean()
cmp_['n'] = opp['gi'].value_counts()
cmp_['Norman dissimilar-phenotype'] = opp.groupby('gi')['genetic_interaction'].apply(
    lambda s: int((s == 'synergy (dissimilar phenotype)').sum()))
print('\nOpposite-pole pairs: what separates synergy from suppression')
cmp_.round(2)

## Biological findings

All statements concern the measured, labelled panel (86 pairs). It is a curated selection, so none is
claimed to extend to unmeasured combinations.

**1. Interaction type is readable from seven model parameters** (accuracy 0.77, balanced 0.78). ρ
separates co-directional from opposing pairs; the size of the combination relative to the sum separates
redundant (smaller), additive (equal) and synergy (larger); private latent factors with undamped parent
scales mark potentiation; decoder dominance marks epistasis; and synergy acting on latent factors that
neither parent gates marks neomorphism.

**2. One dominant latent axis carries the erythroid programme, and the C/EBPs oppose it.** F13
(erythroid score 5.1; reproduced across runs, |cos| 0.86; opened by 87 of 105 perturbations) is pushed
up by IKZF3, DUSP9, TP73 and PRDM1 and down by CEBPA, CEBPE, CEBPB and OSR2. F5 (erythroid 2.8) is
driven by FEV, ELMSAN1, IKZF3 and LHX1. **F12 is the latent axis of the lineage conflict** (granulocytic
+1.6, erythroid −1.3): the C/EBPs, MAPK1 and ETS2 load on one end and CNN1, PRTG, IGDCC3 and KLF1 on the
other — the same genes that form the suppression pairs in point 6.

**3. The fate axis is erythroid up versus erythroid down.** With canonical markers, granulocytic
priming opposes the erythroid programme (r = −0.31). Megakaryocytic priming is a **weak signal** — its
markers are detected in ~1% of control cells and the model cannot decode them — but its score is
reproducible (split-half reliability 0.84, versus 0.99 erythroid and 0.97 granulocytic), and it does not
oppose the erythroid programme: the two co-vary weakly (r = +0.29, 95% CI 0.04–0.50, permutation
p = 0.005; unchanged without FEV). The strongest megakaryocytic inducers (FEV, TP73, IKZF3) also raise
erythroid genes, consistent with a bipotent megakaryocyte–erythroid pattern. The score also co-varies
with granulocytic markers (r = +0.35; CEBPA is among the top megakaryocytic scorers), so megakaryocytic priming is not cleanly
separable from either fate in this screen. The model's decoded erythroid score matches the observed one
(r = 0.80). Norman's erythroid group raises the erythroid programme (+0.39), his granulocyte group
lowers it and raises granulocytic markers (−0.81 / +2.37), and the G1-arrest group (CDKN1A/B/C) sits on
the erythroid side (+0.65).

**4. The learned modules recover Norman's functional clusters.** 9 of his 20 perturbation-map clusters
and phenotype groups fall into a single module more often than random modules of the same sizes would
allow (79% of members in the best module on average, versus 41% by chance). The tyrosine-phosphatase
cluster (6 of 8 in M2), the bZIP TFs (5 of 5 in M3), the granulocyte group (4 of 4 in M3), the CDK
inhibitors (3 of 3 in M7), the RNA-pol-II TFs (7 of 12 in M1), the forkhead TFs (4 of 6 in M1),
regulation of apoptosis (5 of 9 in M5) and Norman's erythroid group (4 of 6 in M2) are all concentrated
(BH q between 0.004 and 0.032 across the 20 clusters). Most unconcentrated clusters have only two members, which cannot
reach significance. The
clusters' own programme scores follow the modules: the phosphatase, erythrocytosis, CDK-inhibitor,
apoptosis and Ig-domain clusters raise the erythroid programme (+0.6 to +0.9), while the bZIP TFs and
granulocyte group raise granulocytic markers (+2.0, +2.4) and MAPK1–ETS2 lowers the erythroid programme
(−1.2).

**5. Modules split into three poles, and the "down" pole is not one fate.** *Erythroid up:* M5
(erythroid + megakaryocytic: FEV, LHX1, IKZF3), M11, M7 (G1 arrest), M2 (Norman's erythroid core CNN1,
PTPN9, PTPN12, UBASH3B) and M1 (CBL). *Neutral:* M4 and M6 (MAP2K3/MAP2K6). *Erythroid down:* M3 is
granulocytic (all four C/EBP/PU.1 members of Norman's granulocyte group); M8 (MAPK1, ETS2), M9, M10 and
M12 lower the erythroid programme without adopting a granulocytic or megakaryocytic signature.

**6. Where two modules sit constrains the interaction; the model's parameters decide among the options.**
- *Redundancy is within a module:* all 8 redundant pairs join two genes of the same module
  (27% of labelled pairs are within-module; q = 4×10⁻⁵), and 5 of the 8 lie in the erythroid-down pole,
  including the C/EBP paralogs (q = 0.003).
- *Suppression needs opposite poles:* no suppression pair is within a module (q = 0.035), and all 12
  join an erythroid-up module to an erythroid-down module (38% base rate; q = 2×10⁻⁵). At the gene level,
  11 of 12 suppression pairs have parents that push the erythroid programme in opposite directions
  (48% of labelled pairs; q = 0.003).
  The antagonists are the C/EBPs and the MAPK1/ETS2 module against the erythroid phosphatase module
  (CNN1, PTPN12, UBASH3B), PRTG/IGDCC3 and LYL1.
- *Synergy concentrates in the erythroid-up pole* (11 of 24; q = 0.035). Opposite-pole pairs that
  synergise instead of suppressing (9, seven of them Norman's dissimilar-phenotype type) have parents much
  closer together on the erythroid axis (gap 0.61 versus 1.74) and are the ones the model still aligns: ρ +0.36 versus −0.13, a combination 1.21× the sum versus 0.91×, and undamped
  versus damped parent scales (1.76 versus 1.29).
- *Additivity avoids opposite poles* (2 of 15; q = 0.04).

All q-values are Benjamini–Hochberg adjusted within their family of tests (the 8 pair-level tests
above; the 20 cluster tests in point 4).

*Caveats.* Latent axes match clearly across runs (every run's total matched similarity beats random pairings, p = 0.0002), but F3 and F15 cannot be matched unambiguously — their best alternative is as similar as their assigned partner — so they are not interpreted. Canonical megakaryocytic markers are detected in only ~1% of control cells, so that score is
noisy; the model cannot decode them at all, and it decodes only four canonical granulocyte genes. Rule cut
points were fit on these labelled pairs.

## Figures

Labelled pairs only. The erythroid axis runs from *erythroid up* (brown) to *erythroid down* (blue);
interaction classes keep their own colours.

In [ ]:
from matplotlib.path import Path as MPath
ERY_CMAP = mpl.colors.LinearSegmentedColormap.from_list('ery', ['#2166ac', '#f7f7f7', '#8c2d04'])
enorm = mpl.colors.TwoSlopeNorm(vmin=fate['erythroid'].min(), vcenter=0, vmax=fate['erythroid'].max())
POLE_C = {'erythroid up': '#8c2d04', 'neutral': '#9e9e9e', 'erythroid down': '#2166ac'}
ID_C = {'granulocytic': '#7a7a00', 'erythroid + megakaryocytic': '#01665e'}
PROG_C = {'erythroid': '#8c2d04', 'granulocytic': '#7a7a00'}
GROUP_C = {'erythroid': '#8c2d04', 'granulocyte': '#7a7a00', 'G1 arrest': '#5e3c99'}

# re-decode the exemplar pairs (figure 6)
EXAMPLES = [('CEBPA', 'CEBPB'), ('CBL', 'PTPN12'), ('CEBPE', 'CNN1'), ('DUSP9', 'MAPK1'), ('FEV', 'ISL2')]
ex_pred = {e: [] for e in EXAMPLES}
for run in range(1, N_RUNS + 1):
    m = slk.tl.load_results(FIG / 'models' / f'run_{run}.pth').to(DEVICE)
    with torch.no_grad():
        sh = m._get_all_pert_shifts(DEVICE)[1:]
        S_, q_ = sh[:, 0], sh[:, 2]
        u = m._abduct_ntc(x_bg)
        def dec(shift):
            z = (u[None] + shift[:, None]).reshape(-1, u.shape[-1])
            return torch.log1p(m._decode_to_expr(z, 1.) * 1e4).reshape(shift.shape[0], N_BG, -1).mean(1)
        base = torch.log1p(m._decode_to_expr(u, 1.) * 1e4).mean(0)
        ia = torch.as_tensor([g2i[a] for a, _ in EXAMPLES], device=DEVICE)
        ib = torch.as_tensor([g2i[b] for _, b in EXAMPLES], device=DEVICE)
        ca, cb = m._parent_scales_from_rho(q_[ia], q_[ib]); syn = m._synergy_shift_from_rho(q_[ia], q_[ib])
        d_a, d_b = dec(S_[ia]) - base, dec(S_[ib]) - base
        d_ab = dec(ca * S_[ia] + cb * S_[ib] + syn) - base
        for k, e in enumerate(EXAMPLES):
            ex_pred[e].append(np.stack([d_a[k].cpu().numpy(), d_b[k].cpu().numpy(), d_ab[k].cpu().numpy()]))
    del m; torch.cuda.empty_cache()
print('decoded', len(EXAMPLES), 'exemplar pairs')

### 1. Position on the erythroid axis and interaction type

**(a)** Modules ordered by their members' observed erythroid score (number under each module). Arcs are
labelled interactions: cooperative classes above the line, suppression / epistasis / neomorphic below;
loops are pairs within one module. Background: brown = *erythroid up* pole, blue = *erythroid down*
pole, white = neutral. A coloured ring marks a module's identity within its pole (olive = granulocytic,
teal = erythroid + megakaryocytic; megakaryocytic is a weak signal, see section 4). **(b)** Every labelled
pair placed by its two parents' observed
erythroid scores. The shaded corner holds pairs whose parents push the erythroid programme in opposite
directions.

### Fig. 5b: labelled pairs as arcs between consensus modules on the erythroid axis

In [ ]:
order_m = mods.sort_values('erythroid').index.tolist()     # erythroid down (left) -> up (right)
ery_sem = fate.groupby(module.reindex(genes).values)['erythroid'].sem()     # spread of the module mean
xr = {m_: i for i, m_ in enumerate(order_m)}
COOP = ['approximately additive', 'synergy', 'redundant', 'potentiation']
fig = plt.figure(figsize=(14, 11))
gs_ = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], width_ratios=[1, .42], hspace=.1, wspace=.05)
ax = fig.add_subplot(gs_[0, :])
for pole, col in POLE_C.items():
    idx = [xr[m_] for m_ in order_m if mods.loc[m_, 'pole'] == pole]
    if idx and pole != 'neutral':
        ax.axvspan(min(idx) - .5, max(idx) + .5, color=col, alpha=.07)
        ax.text((min(idx) + max(idx)) / 2, 6.4, pole, color=col, fontsize=12.2, fontweight='bold', ha='center')
off = dict(zip(CLASSES, np.linspace(-.18, .18, len(CLASSES))))
for (ma, mb, cls), n in lab.groupby(['module_a', 'module_b', 'gi']).size().items():
    xa, xb = sorted([xr[ma], xr[mb]])
    sgn = 1 if cls in COOP else -1
    if xa == xb:
        h = sgn * (0.5 + 0.12 * CLASSES.index(cls))
        verts = [(xa - .12, 0), (xa - .32, h), (xa + .32, h), (xa + .12, 0)]
    else:
        h = sgn * (0.55 * (xb - xa) + 0.3) * (1 + off[cls])
        verts = [(xa, 0), (xa, h), (xb, h), (xb, 0)]
    ax.add_patch(mpl.patches.PathPatch(MPath(verts, [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4]),
                                       fc='none', ec=PALETTE[cls], lw=.8 + 1.4 * n, alpha=.45))
for m_ in order_m:
    r_ = mods.loc[m_]
    ax.scatter(xr[m_], 0, s=230 + 30 * r_['n'], color=ERY_CMAP(enorm(r_['erythroid'])), edgecolor='0.3',
               lw=.6, zorder=5)
    ax.text(xr[m_], 0, m_, ha='center', va='center', fontsize=10.1, fontweight='bold', zorder=6, color='k')
    ax.text(xr[m_], -1.4, f"{r_['erythroid']:+.2f}\n± {ery_sem[m_]:.2f}", ha='center', fontsize=8.1,
            color='0.3', zorder=6, linespacing=.95,
            bbox=dict(fc='white', ec='none', pad=.5, alpha=.8))
ax.axhline(0, color='0.5', lw=.6)
ax.set_xlim(-.7, len(order_m) - .3); ax.set_ylim(-6.9, 6.9); ax.set_xticks([]); ax.set_yticks([])
for s_ in ['left', 'bottom']:
    ax.spines[s_].set_visible(False)
h_cls = [mpl.lines.Line2D([], [], color=PALETTE[c], lw=3, label=short(c)) for c in CLASSES]
ax.legend(handles=h_cls, frameon=False, fontsize=9.5, loc='upper left', bbox_to_anchor=(1.0, 1.0), title='interaction')
_cb = fig.colorbar(mpl.cm.ScalarMappable(norm=enorm, cmap=ERY_CMAP), ax=ax, fraction=.018, pad=.16,
                   location='right', shrink=.5)
_cb.set_label('module mean erythroid score', fontsize=9.5)
_cb.ax.tick_params(labelsize=8.1)
ax.text(-0.02, 1.02, 'a', transform=ax.transAxes, fontsize=15, va='bottom', ha='right')

ax2 = fig.add_subplot(gs_[1, 0])
lo, hi = lab['ery_lo'].min() - .2, lab['ery_hi'].max() + .2
ax2.add_patch(mpl.patches.Rectangle((lo, 0), 0 - lo, hi, color='0.9', zorder=0))
ax2.plot([lo, hi], [lo, hi], color='0.75', lw=.6)
ax2.axvline(0, color='0.6', ls='--', lw=.6); ax2.axhline(0, color='0.6', ls='--', lw=.6)
for c in CLASSES:
    s_ = lab[lab['gi'] == c]
    ax2.scatter(s_['ery_lo'], s_['ery_hi'], s=40, color=PALETTE[c], edgecolor='white', lw=.5, label=short(c), zorder=3)
ax2.text(lo + .05, hi - .05, 'parents push the erythroid\nprogramme in opposite directions', fontsize=9.5, va='top', color='0.3')
ax2.text(.05, hi - .05, 'both raise it', fontsize=9.5, va='top', color=POLE_C['erythroid up'])
xhi = max(lab['ery_lo'].max(), 0) + .35
ax2.text(-.05, lo + .05, 'both lower it', fontsize=9.5, color=POLE_C['erythroid down'], ha='right',
         bbox=dict(fc='white', ec='none', pad=1, alpha=.8))
ax2.set(xlim=(lo, xhi), ylim=(lo, hi), xlabel='observed erythroid score, lower parent',
        ylabel='observed erythroid score, higher parent')
ax2.text(-0.08, 1.02, 'b', transform=ax2.transAxes, fontsize=15, va='bottom', ha='right')
ax2.set_title('Each labelled pair, placed by its two parents', fontsize=12.2, loc='left')

ax3 = fig.add_subplot(gs_[1, 1]); ax3.axis('off')
txt = ['Enrichment on this panel', '(Fisher; BH-adjusted q, 8 tests)']
for _, r_ in claims.iterrows():
    txt.append(f"{r_['class']}: {r_['in']} {r_['where']}\n   (base {r_['base rate']}, q = {r_['q (BH)']:.1g})")
ax3.text(0, 1, '\n'.join(txt), va='top', fontsize=9.9, family='monospace')
plt.show()